# Keypoint Video Inference and Character Overlay on Colab

Colabで動画を読み込み、既定ではYOLO Pose fallbackで人物の17関節を推論します。RF-DETRは `RUN_RFDETR=True` にした場合だけ読み込みます。出力は注釈付き動画、フレームごとのJSONL、関節ごとのCSV、信頼度分布プロット、動画内の人形/色付きキャラクター合成です。

注意: RF-DETR keypoint preview が返すのは関節ごとの `keypoint_confidence` と人物検出の `detection_confidence` です。モデル内部の完全な確率ヒートマップではありません。このノートブックでは、それらの信頼度を確率スコアとして保存・集計します。

## 1. GPU確認と依存関係のインストール

Colabのメニューで `Runtime > Change runtime type > T4 GPU` 以上を選んでから実行してください。T4でも動作確認できます。長い動画や高解像度ならL4/A100の方が快適です。

In [ ]:
!nvidia-smi
import subprocess
import sys

INSTALL_RFDETR = False  # RF-DETRも試す場合だけ True。通常の人物検出/合成はYOLO Poseで進める。
base_packages = ['supervision', 'opencv-python-headless', 'matplotlib', 'tqdm', 'ultralytics']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *base_packages])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas==2.2.2'])
if INSTALL_RFDETR:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'rfdetr'])

## 2. 動画を読み込む

`USE_DRIVE = False` の場合は、セル実行時にローカルPCから動画をアップロードします。Google Drive上の動画を使う場合は `USE_DRIVE = True` にして `VIDEO_PATH` を指定してください。Driveは `force_remount=False` で1回だけ接続し、入力動画は処理用に `/content` へコピーします。

In [ ]:
from pathlib import Path

USE_DRIVE = False
VIDEO_PATH = "/content/drive/MyDrive/input_video.mp4"  # USE_DRIVE=True の場合は Drive 上の動画パスに変更
COPY_DRIVE_INPUT_TO_CONTENT = True  # Drive動画を/contentへコピーして処理を安定・高速化

if USE_DRIVE:
    from google.colab import drive
    drive_mount_path = Path('/content/drive')
    if not drive_mount_path.exists() or not any(drive_mount_path.iterdir()):
        drive.mount('/content/drive', force_remount=False)
    else:
        print('Google Drive is already mounted; reusing existing mount.')
    source_video_path = Path(VIDEO_PATH)
    if not source_video_path.exists():
        raise FileNotFoundError(source_video_path)
    if COPY_DRIVE_INPUT_TO_CONTENT:
        import shutil
        video_path = Path('/content') / source_video_path.name
        if not video_path.exists() or video_path.stat().st_size != source_video_path.stat().st_size:
            print(f'Copying Drive video to local runtime: {video_path}')
            shutil.copy2(source_video_path, video_path)
        else:
            print(f'Using existing local runtime copy: {video_path}')
    else:
        video_path = source_video_path
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("動画ファイルがアップロードされていません。")
    video_path = Path(next(iter(uploaded.keys()))).resolve()

if not video_path.exists():
    raise FileNotFoundError(video_path)

def _video_identity(path):
    path = Path(path)
    stat = path.stat()
    return {
        'path': str(path.resolve()),
        'name': path.name,
        'size': int(stat.st_size),
        'mtime_ns': int(stat.st_mtime_ns),
    }

CURRENT_VIDEO_ID = _video_identity(video_path)
_previous_video_id = globals().get('CURRENT_VIDEO_ID_PREVIOUS')
if _previous_video_id is not None and _previous_video_id != CURRENT_VIDEO_ID:
    stale_names = [
        'frame_records', 'df', 'yolo_frame_records', 'yolo_df', 'motion_payload',
        'character_motion_by_frame', 'rigged_raw_path', 'rigged_final_path',
        'rigged_written', 'last_pose', 'last_overlay_bgra', 'active_pose_source',
        'yolo_source_video_id', 'rfdetr_source_video_id', 'motion_payload_source_video_id',
    ]
    for _name in stale_names:
        globals().pop(_name, None)
    print('Input video changed: cleared stale pose/motion/character outputs. Re-run pose inference before rendering.')
CURRENT_VIDEO_ID_PREVIOUS = dict(CURRENT_VIDEO_ID)
print(f"Input video: {video_path}")
print(f"Input video id: name={CURRENT_VIDEO_ID['name']} size={CURRENT_VIDEO_ID['size']} mtime_ns={CURRENT_VIDEO_ID['mtime_ns']}")

## 3. 設定とモデル読み込み

`FRAME_STRIDE` を大きくすると処理フレーム数を間引けます。まずは `MAX_FRAMES` を小さめにして動作確認してください。

In [ ]:
import json
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

OUTPUT_DIR = Path('/content/rfdetr_keypoint_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_RFDETR = globals().get('RUN_RFDETR', globals().get('INSTALL_RFDETR', False))  # 通常はYOLO Pose fallback。RF-DETRを試す時だけ True
DETECTION_THRESHOLD = 0.15
KEYPOINT_CONF_THRESHOLD = 0.15
FRAME_STRIDE = 1
MAX_FRAMES = None  # 例: 300 にすると先頭300フレームだけ処理
DRAW_LOW_CONFIDENCE = True
OPTIMIZE_FOR_INFERENCE = True  # 検出ゼロが続く場合は False にして切り分け

COCO_KEYPOINT_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle',
]

COCO_SKELETON = [
    (5, 7), (7, 9), (6, 8), (8, 10), (5, 6),
    (5, 11), (6, 12), (11, 12), (11, 13), (13, 15),
    (12, 14), (14, 16), (0, 1), (0, 2), (1, 3), (2, 4),
]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device={device}")
model = None
if RUN_RFDETR:
    from rfdetr import RFDETRKeypointPreview
    model = RFDETRKeypointPreview(device=device)
    if device == 'cuda' and OPTIMIZE_FOR_INFERENCE:
        print('Optimizing model for FP16 GPU inference...')
        model.optimize_for_inference(dtype=torch.float16)
        print('Model optimization complete.')
else:
    class _SkippedRFDETRKeypoints:
        xy = np.empty((0, 17, 2), dtype=np.float32)
        keypoint_confidence = np.empty((0, 17), dtype=np.float32)
        detection_confidence = np.empty((0,), dtype=np.float32)
        data = {'xyxy': np.empty((0, 4), dtype=np.float32)}

    class _SkippedRFDETRModel:
        def predict(self, *args, **kwargs):
            return _SkippedRFDETRKeypoints()

    model = _SkippedRFDETRModel()
    print('RUN_RFDETR=False: RF-DETR model loading is skipped. Use ## 5 YOLO Pose fallback, then downstream cells.')

## 4. 検出デバッグ

本処理の前に、数フレームだけ低しきい値で推論して、人物が検出されるか確認します。ここで `detections=0` が続く場合は、モデル初期化・動画の向き・人物サイズ・暗さ・しきい値を疑います。

In [ ]:
from IPython.display import display
from PIL import Image

DEBUG_THRESHOLDS = [0.01, 0.05, 0.10, 0.15, 0.25]
DEBUG_SAMPLE_FRAMES = 5

def _count_keypoints(key_points):
    xy = getattr(key_points, 'xy', None)
    return 0 if xy is None else len(xy)


cap_debug = cv2.VideoCapture(str(video_path))
if not cap_debug.isOpened():
    raise RuntimeError(f"動画を開けません: {video_path}")

debug_total = int(cap_debug.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
if debug_total > 0:
    debug_frame_indices = np.linspace(0, max(0, debug_total - 1), DEBUG_SAMPLE_FRAMES, dtype=int).tolist()
else:
    debug_frame_indices = list(range(DEBUG_SAMPLE_FRAMES))

debug_rows = []
for debug_frame_idx in debug_frame_indices:
    cap_debug.set(cv2.CAP_PROP_POS_FRAMES, int(debug_frame_idx))
    ok, frame_bgr = cap_debug.read()
    if not ok:
        continue
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    for threshold in DEBUG_THRESHOLDS:
        key_points = model.predict(frame_rgb, threshold=threshold, include_source_image=False)
        count = _count_keypoints(key_points)
        det_conf = getattr(key_points, 'detection_confidence', None)
        max_conf = float(np.max(det_conf)) if det_conf is not None and len(det_conf) else None
        debug_rows.append({
            'frame_index': int(debug_frame_idx),
            'threshold': float(threshold),
            'detections': int(count),
            'max_detection_confidence': max_conf,
        })

cap_debug.release()
debug_df = pd.DataFrame(debug_rows)
display(debug_df)

best_debug = debug_df.sort_values(['detections', 'max_detection_confidence'], ascending=[False, False], na_position='last').head(1)
if not best_debug.empty and int(best_debug.iloc[0]['detections']) > 0:
    recommended = float(best_debug.iloc[0]['threshold'])
    print(f"検出あり。まず DETECTION_THRESHOLD={recommended:.2f} 付近で本処理してください。")
else:
    print('RF-DETRでは低しきい値でも検出がありません。')
    print('次は「## 5. YOLO Pose fallbackで動画をフレーム処理する」を実行してください。')
    print('その後、「## 6. RF-DETRで動画をフレーム処理する」はスキップして、動画変換・分布・3D人形・動画内合成へ進みます。')

## 5. YOLO Pose fallbackで動画をフレーム処理する

RF-DETRで `detections=0` が続く動画向けの代替ルートです。複数人物のダンス動画では、まずこのセルを使ってください。このセルは後段と同じ `frame_records` / `df` / `raw_output_video` を作るので、以降の可視化・3D人形・動画内合成セルをそのまま実行できます。

In [ ]:
from ultralytics import YOLO

YOLO_MODEL_NAME = 'yolov8m-pose.pt'  # 速さ優先: yolov8n-pose.pt / 精度優先: yolov8x-pose.pt
YOLO_CONF_THRESHOLD = 0.15
YOLO_IOU_THRESHOLD = 0.50
YOLO_IMGSZ = 960
YOLO_USE_TRACKING = globals().get('YOLO_USE_TRACKING', True)
YOLO_TRACKER_CONFIG = globals().get('YOLO_TRACKER_CONFIG', 'bytetrack.yaml')

yolo_pose = YOLO(YOLO_MODEL_NAME)

def _to_numpy(value, shape=None, dtype=np.float32):
    if value is None:
        if shape is None:
            return None
        return np.empty(shape, dtype=dtype)
    if hasattr(value, 'detach'):
        value = value.detach().cpu().numpy()
    array = np.asarray(value)
    if shape is not None and array.size == 0:
        return np.empty(shape, dtype=dtype)
    return array.astype(dtype, copy=False)


def draw_yolo_keypoints(frame_bgr, xy, kp_conf, boxes, det_conf, track_ids=None):
    annotated = frame_bgr.copy()
    for person_idx in range(len(xy)):
        color = (40, 220, 120)
        if person_idx < len(boxes):
            x1, y1, x2, y2 = boxes[person_idx].astype(int).tolist()
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            if track_ids is not None and person_idx < len(track_ids) and track_ids[person_idx] is not None:
                label = f"track {track_ids[person_idx]} {det_conf[person_idx]:.2f}" if person_idx < len(det_conf) else f"track {track_ids[person_idx]}"
            else:
                label = f"person {det_conf[person_idx]:.2f}" if person_idx < len(det_conf) else "person"
            cv2.putText(annotated, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        visible = kp_conf[person_idx] >= KEYPOINT_CONF_THRESHOLD if person_idx < len(kp_conf) else np.ones(len(xy[person_idx]), dtype=bool)
        for a, b in COCO_SKELETON:
            if a >= len(xy[person_idx]) or b >= len(xy[person_idx]):
                continue
            if not DRAW_LOW_CONFIDENCE and (not visible[a] or not visible[b]):
                continue
            ax, ay = xy[person_idx, a].astype(int).tolist()
            bx, by = xy[person_idx, b].astype(int).tolist()
            cv2.line(annotated, (ax, ay), (bx, by), (255, 180, 40), 2)

        for joint_idx, (x, y) in enumerate(xy[person_idx]):
            conf = float(kp_conf[person_idx, joint_idx]) if person_idx < len(kp_conf) else 1.0
            if not DRAW_LOW_CONFIDENCE and conf < KEYPOINT_CONF_THRESHOLD:
                continue
            radius = 4 if conf >= KEYPOINT_CONF_THRESHOLD else 2
            point_color = (40, 220, 120) if conf >= KEYPOINT_CONF_THRESHOLD else (140, 140, 140)
            cv2.circle(annotated, (int(x), int(y)), radius, point_color, -1)
    return annotated


cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f"動画を開けません: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
if MAX_FRAMES is not None:
    total_for_progress = min(total_frames, MAX_FRAMES)
else:
    total_for_progress = total_frames if total_frames > 0 else None

raw_output_video = OUTPUT_DIR / 'annotated_keypoints_yolo_raw.mp4'
final_output_video = OUTPUT_DIR / 'annotated_keypoints_yolo.mp4'
jsonl_path = OUTPUT_DIR / 'frame_keypoints_yolo.jsonl'
csv_path = OUTPUT_DIR / 'joint_confidences_yolo.csv'

writer = cv2.VideoWriter(
    str(raw_output_video),
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps / max(FRAME_STRIDE, 1),
    (width, height),
)
if not writer.isOpened():
    raise RuntimeError("出力動画のVideoWriterを開けません。")

rows = []
frame_records = []
processed = 0
frame_idx = -1
device_arg = 0 if torch.cuda.is_available() else 'cpu'

with jsonl_path.open('w', encoding='utf-8') as jsonl:
    progress = tqdm(total=total_for_progress, desc='YOLO pose inference')
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frame_idx += 1
        if frame_idx % FRAME_STRIDE != 0:
            continue
        if MAX_FRAMES is not None and processed >= MAX_FRAMES:
            break

        if YOLO_USE_TRACKING:
            result = yolo_pose.track(
                frame_bgr,
                conf=YOLO_CONF_THRESHOLD,
                iou=YOLO_IOU_THRESHOLD,
                imgsz=YOLO_IMGSZ,
                device=device_arg,
                persist=True,
                tracker=YOLO_TRACKER_CONFIG,
                verbose=False,
            )[0]
        else:
            result = yolo_pose.predict(
                frame_bgr,
                conf=YOLO_CONF_THRESHOLD,
                iou=YOLO_IOU_THRESHOLD,
                imgsz=YOLO_IMGSZ,
                device=device_arg,
                verbose=False,
            )[0]

        xy = _to_numpy(result.keypoints.xy if result.keypoints is not None else None, shape=(0, 17, 2))
        kp_conf = _to_numpy(result.keypoints.conf if result.keypoints is not None else None, shape=(len(xy), 17))
        boxes = _to_numpy(result.boxes.xyxy if result.boxes is not None else None, shape=(len(xy), 4))
        det_conf = _to_numpy(result.boxes.conf if result.boxes is not None else None, shape=(len(xy),))
        track_raw = result.boxes.id if result.boxes is not None and getattr(result.boxes, 'id', None) is not None else None
        track_ids_np = _to_numpy(track_raw, shape=(0,), dtype=np.int32) if track_raw is not None else np.empty((0,), dtype=np.int32)
        track_ids = [int(track_ids_np[i]) if i < len(track_ids_np) else None for i in range(len(xy))]

        annotated = draw_yolo_keypoints(frame_bgr, xy, kp_conf, boxes, det_conf, track_ids)
        writer.write(annotated)

        frame_payload = {
            'frame_index': int(frame_idx),
            'time_s': float(frame_idx / fps),
            'people': [],
        }

        for person_idx in range(len(xy)):
            person_payload = {
                'person_index': int(person_idx),
                'track_id': track_ids[person_idx] if person_idx < len(track_ids) else None,
                'detection_confidence': float(det_conf[person_idx]) if person_idx < len(det_conf) else None,
                'bbox_xyxy': boxes[person_idx].astype(float).tolist() if person_idx < len(boxes) else None,
                'keypoints': [],
            }
            for joint_idx, joint_xy in enumerate(xy[person_idx]):
                joint_name = COCO_KEYPOINT_NAMES[joint_idx] if joint_idx < len(COCO_KEYPOINT_NAMES) else f'keypoint_{joint_idx}'
                confidence = float(kp_conf[person_idx, joint_idx]) if person_idx < len(kp_conf) else math.nan
                visible = bool(confidence >= KEYPOINT_CONF_THRESHOLD) if not math.isnan(confidence) else False
                item = {
                    'joint_index': int(joint_idx),
                    'joint_name': joint_name,
                    'x': float(joint_xy[0]),
                    'y': float(joint_xy[1]),
                    'keypoint_confidence': confidence,
                    'visible': visible,
                }
                person_payload['keypoints'].append(item)
                rows.append({
                    'frame_index': int(frame_idx),
                    'time_s': float(frame_idx / fps),
                    'person_index': int(person_idx),
                    'track_id': person_payload.get('track_id'),
                    'detection_confidence': person_payload['detection_confidence'],
                    **item,
                })
            frame_payload['people'].append(person_payload)

        jsonl.write(json.dumps(frame_payload, ensure_ascii=False) + '\n')
        frame_records.append(frame_payload)
        processed += 1
        progress.update(1)

    progress.close()

cap.release()
writer.release()

df = pd.DataFrame(rows)
df.to_csv(csv_path, index=False)

# Keep YOLO results safe even if RF-DETR cells are executed later by accident.
yolo_frame_records = frame_records
yolo_df = df
yolo_raw_output_video = raw_output_video
yolo_final_output_video = final_output_video
yolo_jsonl_path = jsonl_path
yolo_csv_path = csv_path
yolo_source_video_id = dict(globals().get('CURRENT_VIDEO_ID', _video_identity(video_path) if '_video_identity' in globals() else {'path': str(video_path)}))
active_pose_source = 'yolo'

effective_video_fps = fps / max(FRAME_STRIDE, 1)
person_instances = sum(len(frame.get('people', [])) for frame in frame_records)
avg_people_per_frame = person_instances / processed if processed else 0.0
print(f"processed_frames={processed}")
print(f"person_frame_instances={person_instances}")
track_ids_seen = sorted({int(p['track_id']) for frame in frame_records for p in frame.get('people', []) if p.get('track_id') is not None})
print(f"avg_people_per_frame={avg_people_per_frame:.2f}")
print(f"YOLO tracking={'on' if YOLO_USE_TRACKING else 'off'} tracker={YOLO_TRACKER_CONFIG} track_ids={track_ids_seen[:20]} total_tracks={len(track_ids_seen)}")
if track_ids_seen:
    print('Target selection tip: set CHARACTER_SOURCE_TRACK_ID to one of these track IDs in the colored character cell.')
else:
    print('No YOLO track IDs were produced. Keep CHARACTER_SOURCE_TRACK_ID=None and use CHARACTER_SOURCE_PERSON fallback, or rerun with YOLO_USE_TRACKING=True.')
print(f"joint_rows={len(df)}")
print(f"source_fps={fps:.2f} effective_output_fps={effective_video_fps:.2f}")
print(f"raw_output_video={raw_output_video}")
print(f"jsonl={jsonl_path}")
print(f"csv={csv_path}")
df.head()

## 5b. MediaPipe 3D姿勢推定（選択人物のみ）

YOLOのトラッキングbboxで対象ダンサーを切り抜き、MediaPipe Poseで33ランドマークの3D座標(x, y, z)を推定します。zはカメラに近いほど負の値です。ここで作った `mp3d_by_frame` は後段のキャラクター合成セルが自動利用し、深度ソート描画(手が顔の前/後ろを実際の奥行きで判定)と腰のくねり表現が有効になります。`RUN_MP3D=False` でスキップした場合は従来の2Dキーポイント描画で動きます。


In [ ]:
# --- MediaPipe 3D pose for the selected person / 選択人物のMediaPipe 3D姿勢推定 ---
# 現行のMediaPipe Tasks API (PoseLandmarker) を使用します。旧 solutions API は
# 新しいランタイムでは削除されているため使いません。
# YOLOのbboxで対象人物を切り抜き、33ランドマークの3D座標を取得します。
# 出力: mp3d_by_frame = {frame_index: {'landmarks': {関節名: [x_px, y_px, z_px, visibility]}}}
# zはカメラに近いほど負。後段の合成セルが自動的に利用します。
import urllib.request
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm


def _ensure_mediapipe():
    import importlib
    import subprocess
    import sys
    try:
        import mediapipe as mp_lib
        from mediapipe.tasks.python import vision as _v  # noqa: F401
        return mp_lib
    except (ImportError, AttributeError):
        pass
    print('mediapipe をインストール/更新します...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'mediapipe'], check=True)
    importlib.invalidate_caches()
    for _name in [m for m in list(sys.modules) if m == 'mediapipe' or m.startswith('mediapipe.')]:
        sys.modules.pop(_name, None)
    try:
        import mediapipe as mp_lib
        from mediapipe.tasks.python import vision as _v  # noqa: F401
        return mp_lib
    except (ImportError, AttributeError) as exc:
        raise RuntimeError(
            'mediapipe を読み込めませんでした。Colabメニューの「ランタイム → セッションを再起動」後、'
            'このセルから再実行してください。'
        ) from exc


mp_lib = _ensure_mediapipe()
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.core.base_options import BaseOptions

RUN_MP3D = globals().get('RUN_MP3D', True)
# モデル: 'heavy'(最高精度・重い) / 'full'(バランス) / 'lite'(高速)。品質優先なら heavy。
MP3D_MODEL_VARIANT = globals().get('MP3D_MODEL_VARIANT', 'heavy')
# 'video' = フレーム間トラッキングあり(推奨・速い)。'image' = 毎フレーム独立検出(頑健だが遅い)。
MP3D_RUNNING_MODE = globals().get('MP3D_RUNNING_MODE', 'video')
MP3D_MIN_DETECTION_CONFIDENCE = globals().get('MP3D_MIN_DETECTION_CONFIDENCE', 0.4)
MP3D_MIN_TRACKING_CONFIDENCE = globals().get('MP3D_MIN_TRACKING_CONFIDENCE', 0.4)
# 人物bbox周囲の余白(bbox寸法に対する割合)。手足がbboxから大きく出る動画では増やします。
MP3D_CROP_PAD_FRACTION = globals().get('MP3D_CROP_PAD_FRACTION', 0.30)
# 推定対象。Noneなら CHARACTER_SOURCE_TRACK_ID (未定義なら最大bboxの人物) を使います。
# 合成セルと同じ人物になるよう、track IDを使う場合は両方のセルの前に設定してください。
MP3D_SOURCE_TRACK_ID = globals().get('MP3D_SOURCE_TRACK_ID', globals().get('CHARACTER_SOURCE_TRACK_ID', None))

# MediaPipeランドマーク番号 → 既存パイプラインのCOCO関節名
MP_TO_COCO = {
    0: 'nose', 2: 'left_eye', 5: 'right_eye', 7: 'left_ear', 8: 'right_ear',
    11: 'left_shoulder', 12: 'right_shoulder', 13: 'left_elbow', 14: 'right_elbow',
    15: 'left_wrist', 16: 'right_wrist', 23: 'left_hip', 24: 'right_hip',
    25: 'left_knee', 26: 'right_knee', 27: 'left_ankle', 28: 'right_ankle',
}

_mp3d_records = globals().get('yolo_frame_records') or globals().get('frame_records')

if not RUN_MP3D:
    print('RUN_MP3D=False: MediaPipe 3D姿勢推定をスキップします。合成セルは2Dキーポイントで動きます。')
elif not _mp3d_records:
    raise RuntimeError('YOLO/RF-DETRの解析結果がありません。先に姿勢推定セルを実行してください。')
else:
    # PoseLandmarkerモデルのダウンロード(初回のみ)
    _variant = str(MP3D_MODEL_VARIANT).lower()
    if _variant not in ('heavy', 'full', 'lite'):
        raise ValueError(f"MP3D_MODEL_VARIANT は 'heavy'/'full'/'lite' のいずれかです: {_variant}")
    _model_dir = Path('/content') if Path('/content').exists() else Path(globals().get('OUTPUT_DIR', '.'))
    _model_path = _model_dir / f'pose_landmarker_{_variant}.task'
    if not _model_path.exists():
        _model_url = (
            'https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
            f'pose_landmarker_{_variant}/float16/latest/pose_landmarker_{_variant}.task'
        )
        print(f'PoseLandmarkerモデルをダウンロードします: {_model_url}')
        urllib.request.urlretrieve(_model_url, str(_model_path))
        print(f'saved: {_model_path} ({_model_path.stat().st_size / 1024 / 1024:.1f} MB)')

    def _mp3d_bbox_area(person):
        bbox = person.get('bbox_xyxy') if person else None
        if not bbox or len(bbox) < 4:
            return 0.0
        return max(0.0, float(bbox[2]) - float(bbox[0])) * max(0.0, float(bbox[3]) - float(bbox[1]))

    def _mp3d_pick_person(frame_motion):
        people = frame_motion.get('people', []) if frame_motion else []
        if not people:
            return None
        if MP3D_SOURCE_TRACK_ID is not None:
            wanted = int(MP3D_SOURCE_TRACK_ID)
            return next((p for p in people if p.get('track_id') == wanted), None)
        return max(people, key=_mp3d_bbox_area)

    _frames_by_idx = {int(f['frame_index']): f for f in _mp3d_records}
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'動画を開けません: {video_path}')
    _fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    _fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    _fps = float(globals().get('fps', cap.get(cv2.CAP_PROP_FPS) or 30.0))

    _use_video_mode = str(MP3D_RUNNING_MODE).lower() == 'video'
    _options = mp_vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(_model_path)),
        running_mode=mp_vision.RunningMode.VIDEO if _use_video_mode else mp_vision.RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=float(MP3D_MIN_DETECTION_CONFIDENCE),
        min_tracking_confidence=float(MP3D_MIN_TRACKING_CONFIDENCE),
    )
    _landmarker = mp_vision.PoseLandmarker.create_from_options(_options)

    mp3d_by_frame = {}
    mp3d_missing_frames = 0
    try:
        _frame_idx = -1
        progress = tqdm(total=len(_frames_by_idx), desc=f'MediaPipe 3D pose ({_variant})')
        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                break
            _frame_idx += 1
            if _frame_idx not in _frames_by_idx:
                continue
            progress.update(1)
            person = _mp3d_pick_person(_frames_by_idx[_frame_idx])
            bbox = person.get('bbox_xyxy') if person else None
            if not bbox or len(bbox) < 4:
                mp3d_missing_frames += 1
                continue
            x1, y1, x2, y2 = [float(v) for v in bbox[:4]]
            pad_x = (x2 - x1) * float(MP3D_CROP_PAD_FRACTION)
            pad_y = (y2 - y1) * float(MP3D_CROP_PAD_FRACTION)
            cx1 = int(max(0, x1 - pad_x))
            cy1 = int(max(0, y1 - pad_y))
            cx2 = int(min(_fw, x2 + pad_x))
            cy2 = int(min(_fh, y2 + pad_y))
            if cx2 - cx1 < 8 or cy2 - cy1 < 8:
                mp3d_missing_frames += 1
                continue
            crop = frame_bgr[cy1:cy2, cx1:cx2]
            rgb = np.ascontiguousarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            mp_image = mp_lib.Image(image_format=mp_lib.ImageFormat.SRGB, data=rgb)
            if _use_video_mode:
                _ts_ms = int(round(_frame_idx * 1000.0 / max(_fps, 1e-6)))
                result = _landmarker.detect_for_video(mp_image, _ts_ms)
            else:
                result = _landmarker.detect(mp_image)
            if not result.pose_landmarks:
                mp3d_missing_frames += 1
                continue
            lms = result.pose_landmarks[0]
            cw = float(cx2 - cx1)
            ch = float(cy2 - cy1)
            landmarks = {}
            for mp_idx, name in MP_TO_COCO.items():
                lm = lms[mp_idx]
                vis = float(getattr(lm, 'visibility', 1.0) or 0.0)
                landmarks[name] = [
                    float(lm.x) * cw + cx1,   # フル解像度x(px)
                    float(lm.y) * ch + cy1,   # フル解像度y(px)
                    float(lm.z) * cw,          # 深度(pxスケール, 負=カメラ側)
                    vis,
                ]
            mp3d_by_frame[_frame_idx] = {'landmarks': landmarks}
        progress.close()
    finally:
        _landmarker.close()
        cap.release()

    mp3d_source_video_id = dict(globals().get('CURRENT_VIDEO_ID', {}))
    print(f'mp3d_frames={len(mp3d_by_frame)} missing_frames={mp3d_missing_frames}')
    print(f'model={_variant} running_mode={MP3D_RUNNING_MODE} crop_pad={MP3D_CROP_PAD_FRACTION} track_id={MP3D_SOURCE_TRACK_ID}')
    if mp3d_by_frame:
        print('OK: 合成セルが深度ソート描画と腰のくねり表現を自動的に有効化します。')
    else:
        print('警告: MediaPipeで姿勢を推定できたフレームがありません。合成セルは2Dキーポイントで動きます。')


## 6. RF-DETRで動画をフレーム処理する

RF-DETRで検出できる動画向けの元ルートです。検出デバッグで `detections=0` が続く場合、このセルはスキップして上のYOLO Pose fallbackセルを使ってください。

In [ ]:
if not globals().get('RUN_RFDETR', False):
    print('RUN_RFDETR=False: ## 6 RF-DETR本処理は実行せず、YOLO Pose fallback の結果を後段で使います。')
    if 'yolo_frame_records' in globals():
        frame_records = yolo_frame_records
        df = yolo_df
        raw_output_video = yolo_raw_output_video
        final_output_video = yolo_final_output_video
        active_pose_source = 'yolo'
        print(f'YOLO結果を使用中: frames={len(frame_records)} raw_output_video={raw_output_video}')
    else:
        print('まだYOLO結果がありません。先に ## 5 YOLO Pose fallbackで動画をフレーム処理する を実行してください。')
else:
    def _as_numpy(value, shape=None, dtype=np.float32):
        if value is None:
            if shape is None:
                return None
            return np.empty(shape, dtype=dtype)
        array = np.asarray(value)
        if shape is not None and array.size == 0:
            return np.empty(shape, dtype=dtype)
        return array.astype(dtype, copy=False)
    
    
    def draw_keypoints(frame_bgr, key_points):
        annotated = frame_bgr.copy()
        xy = _as_numpy(getattr(key_points, 'xy', None), shape=(0, 17, 2))
        kp_conf = _as_numpy(getattr(key_points, 'keypoint_confidence', None), shape=(len(xy), xy.shape[1] if xy.ndim == 3 else 17))
        det_conf = _as_numpy(getattr(key_points, 'detection_confidence', None), shape=(len(xy),))
        boxes = None
        if hasattr(key_points, 'data'):
            boxes = key_points.data.get('xyxy')
        boxes = _as_numpy(boxes, shape=(len(xy), 4))
    
        for person_idx in range(len(xy)):
            color = (40, 220, 120)
            if person_idx < len(boxes):
                x1, y1, x2, y2 = boxes[person_idx].astype(int).tolist()
                cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
                label = f"person {det_conf[person_idx]:.2f}" if person_idx < len(det_conf) else "person"
                cv2.putText(annotated, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    
            visible = kp_conf[person_idx] >= KEYPOINT_CONF_THRESHOLD if person_idx < len(kp_conf) else np.ones(len(xy[person_idx]), dtype=bool)
            for a, b in COCO_SKELETON:
                if a >= len(xy[person_idx]) or b >= len(xy[person_idx]):
                    continue
                if not DRAW_LOW_CONFIDENCE and (not visible[a] or not visible[b]):
                    continue
                ax, ay = xy[person_idx, a].astype(int).tolist()
                bx, by = xy[person_idx, b].astype(int).tolist()
                cv2.line(annotated, (ax, ay), (bx, by), (255, 180, 40), 2)
    
            for joint_idx, (x, y) in enumerate(xy[person_idx]):
                conf = float(kp_conf[person_idx, joint_idx]) if person_idx < len(kp_conf) else 1.0
                if not DRAW_LOW_CONFIDENCE and conf < KEYPOINT_CONF_THRESHOLD:
                    continue
                radius = 4 if conf >= KEYPOINT_CONF_THRESHOLD else 2
                point_color = (40, 220, 120) if conf >= KEYPOINT_CONF_THRESHOLD else (140, 140, 140)
                cv2.circle(annotated, (int(x), int(y)), radius, point_color, -1)
    
        return annotated
    
    
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"動画を開けません: {video_path}")
    
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if MAX_FRAMES is not None:
        total_for_progress = min(total_frames, MAX_FRAMES)
    else:
        total_for_progress = total_frames if total_frames > 0 else None
    
    raw_output_video = OUTPUT_DIR / 'annotated_keypoints_raw.mp4'
    final_output_video = OUTPUT_DIR / 'annotated_keypoints.mp4'
    jsonl_path = OUTPUT_DIR / 'frame_keypoints.jsonl'
    csv_path = OUTPUT_DIR / 'joint_confidences.csv'
    
    writer = cv2.VideoWriter(
        str(raw_output_video),
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps / max(FRAME_STRIDE, 1),
        (width, height),
    )
    if not writer.isOpened():
        raise RuntimeError("出力動画のVideoWriterを開けません。")
    
    rows = []
    frame_records = []
    processed = 0
    frame_idx = -1
    
    with jsonl_path.open('w', encoding='utf-8') as jsonl:
        progress = tqdm(total=total_for_progress, desc='RF-DETR keypoint inference')
        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                break
            frame_idx += 1
            if frame_idx % FRAME_STRIDE != 0:
                continue
            if MAX_FRAMES is not None and processed >= MAX_FRAMES:
                break
    
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            key_points = model.predict(frame_rgb, threshold=DETECTION_THRESHOLD, include_source_image=False)
            annotated = draw_keypoints(frame_bgr, key_points)
            writer.write(annotated)
    
            xy = _as_numpy(getattr(key_points, 'xy', None), shape=(0, 17, 2))
            kp_conf = _as_numpy(getattr(key_points, 'keypoint_confidence', None), shape=(len(xy), xy.shape[1] if xy.ndim == 3 else 17))
            det_conf = _as_numpy(getattr(key_points, 'detection_confidence', None), shape=(len(xy),))
            boxes = key_points.data.get('xyxy') if hasattr(key_points, 'data') else None
            boxes = _as_numpy(boxes, shape=(len(xy), 4))
    
            frame_payload = {
                'frame_index': int(frame_idx),
                'time_s': float(frame_idx / fps),
                'people': [],
            }
    
            for person_idx in range(len(xy)):
                person_payload = {
                    'person_index': int(person_idx),
                    'detection_confidence': float(det_conf[person_idx]) if person_idx < len(det_conf) else None,
                    'bbox_xyxy': boxes[person_idx].astype(float).tolist() if person_idx < len(boxes) else None,
                    'keypoints': [],
                }
                for joint_idx, joint_xy in enumerate(xy[person_idx]):
                    joint_name = COCO_KEYPOINT_NAMES[joint_idx] if joint_idx < len(COCO_KEYPOINT_NAMES) else f'keypoint_{joint_idx}'
                    confidence = float(kp_conf[person_idx, joint_idx]) if person_idx < len(kp_conf) else math.nan
                    visible = bool(confidence >= KEYPOINT_CONF_THRESHOLD) if not math.isnan(confidence) else False
                    item = {
                        'joint_index': int(joint_idx),
                        'joint_name': joint_name,
                        'x': float(joint_xy[0]),
                        'y': float(joint_xy[1]),
                        'keypoint_confidence': confidence,
                        'visible': visible,
                    }
                    person_payload['keypoints'].append(item)
                    rows.append({
                        'frame_index': int(frame_idx),
                        'time_s': float(frame_idx / fps),
                        'person_index': int(person_idx),
                        'detection_confidence': person_payload['detection_confidence'],
                        **item,
                    })
                frame_payload['people'].append(person_payload)
    
            jsonl.write(json.dumps(frame_payload, ensure_ascii=False) + '\n')
            frame_records.append(frame_payload)
            processed += 1
            progress.update(1)
    
        progress.close()
    
    cap.release()
    writer.release()
    
    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    rfdetr_source_video_id = dict(globals().get('CURRENT_VIDEO_ID', _video_identity(video_path) if '_video_identity' in globals() else {'path': str(video_path)}))
    active_pose_source = 'rfdetr'
    
    effective_video_fps = fps / max(FRAME_STRIDE, 1)
    print(f"processed_frames={processed}")
    print(f"source_fps={fps:.2f} effective_output_fps={effective_video_fps:.2f}")
    print("tqdmの it/s は processed frames per second です。例: 4.45it/s は1秒に約4.45フレーム処理。")
    print(f"raw_output_video={raw_output_video}")
    print(f"jsonl={jsonl_path}")
    print(f"csv={csv_path}")
    df.head()

## 7. Colab表示用に動画を変換する

OpenCVの `mp4v` 出力がブラウザで再生できない場合があるため、ffmpegでH.264に変換します。

In [ ]:
preview_raw_output_video = globals().get('yolo_raw_output_video', raw_output_video)
preview_final_output_video = globals().get('yolo_final_output_video', final_output_video)
print(f"converting={preview_raw_output_video}")
!ffmpeg -y -loglevel error -i "{preview_raw_output_video}" -i "{video_path}" -map 0:v:0 -map 1:a:0? -vcodec libx264 -pix_fmt yuv420p -c:a aac -shortest "{preview_final_output_video}"

from IPython.display import HTML, display
from base64 import b64encode

video_bytes = preview_final_output_video.read_bytes()
video_b64 = b64encode(video_bytes).decode('ascii')
display(HTML(f'''<video width="720" controls><source src="data:video/mp4;base64,{video_b64}" type="video/mp4"></video>'''))
print(preview_final_output_video)

## 8. 関節ごとの信頼度分布を出す

CSVは1行が「1フレーム x 1人物 x 1関節」です。`keypoint_confidence` が関節ごとの確率スコア相当です。

In [ ]:
if df.empty:
    print('検出結果がありません。DETECTION_THRESHOLDを下げるか、人物が映っている動画を使ってください。')
else:
    summary = (
        df.groupby('joint_name')['keypoint_confidence']
        .agg(['count', 'mean', 'std', 'min', 'max'])
        .reset_index()
        .sort_values('mean', ascending=False)
    )
    summary_path = OUTPUT_DIR / 'joint_confidence_summary.csv'
    summary.to_csv(summary_path, index=False)
    display(summary)

    plt.figure(figsize=(12, 5))
    ordered = summary.sort_values('mean', ascending=True)
    plt.barh(ordered['joint_name'], ordered['mean'], xerr=ordered['std'].fillna(0), color='#2f80ed')
    plt.xlim(0, 1)
    plt.xlabel('mean keypoint confidence')
    plt.title('Joint confidence distribution summary')
    plt.tight_layout()
    bar_path = OUTPUT_DIR / 'joint_confidence_bar.png'
    plt.savefig(bar_path, dpi=160)
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.hist(df['keypoint_confidence'].dropna(), bins=30, range=(0, 1), color='#27ae60', alpha=0.85)
    plt.xlabel('keypoint confidence')
    plt.ylabel('count')
    plt.title('All keypoint confidence histogram')
    plt.tight_layout()
    hist_path = OUTPUT_DIR / 'keypoint_confidence_histogram.png'
    plt.savefig(hist_path, dpi=160)
    plt.show()

    print(f"summary={summary_path}")
    print(f"bar_plot={bar_path}")
    print(f"histogram={hist_path}")

## 9. 3Dキャラクター取り込み用モーションJSONを作る

RF-DETRのキーポイントは単眼2D推定なので、厳密な奥行きは含みません。このセルでは、Three.js / Unity / Blender 側でIKやリターゲットに渡しやすいように、画面中心基準の正規化座標とHumanoid向けの中間点を含む `character_motion.json` を作ります。`z` は初期値 `0.0` の2.5Dプレースホルダーです。奥行きまで動かしたい場合は、このJSONを入力にして MotionBERT / VideoPose3D / MediaPipe 3D Pose などを追加してください。

In [ ]:
def _valid_joint(joint):
    return joint is not None and joint.get('visible') and not math.isnan(float(joint.get('keypoint_confidence', math.nan)))


def _normalize_point(x, y, frame_width, frame_height):
    scale = float(max(frame_width, frame_height))
    return {
        'x': float((x - frame_width / 2.0) / scale),
        'y': float(-(y - frame_height / 2.0) / scale),
        'z': 0.0,
    }


def _average_landmark(name, joints_by_name, source_names, frame_width, frame_height):
    points = [joints_by_name.get(source_name) for source_name in source_names]
    points = [point for point in points if _valid_joint(point)]
    if not points:
        return None
    x = float(np.mean([point['x'] for point in points]))
    y = float(np.mean([point['y'] for point in points]))
    confidence = float(np.mean([point['keypoint_confidence'] for point in points]))
    position = _normalize_point(x, y, frame_width, frame_height)
    return {
        'name': name,
        'position': position,
        'confidence': confidence,
        'visible': True,
        'source_joints': source_names,
    }


def _single_joint_landmark(name, joints_by_name, source_name, frame_width, frame_height):
    joint = joints_by_name.get(source_name)
    if not _valid_joint(joint):
        return None
    return {
        'name': name,
        'position': _normalize_point(joint['x'], joint['y'], frame_width, frame_height),
        'confidence': float(joint['keypoint_confidence']),
        'visible': True,
        'source_joints': [source_name],
    }


def _build_humanoid_landmarks(person, frame_width, frame_height):
    joints_by_name = {item['joint_name']: item for item in person.get('keypoints', [])}
    landmarks = {}
    specs = [
        ('hips', ['left_hip', 'right_hip']),
        ('shoulders', ['left_shoulder', 'right_shoulder']),
        ('head', ['nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear']),
    ]
    for name, source_names in specs:
        landmark = _average_landmark(name, joints_by_name, source_names, frame_width, frame_height)
        if landmark:
            landmarks[name] = landmark

    joint_specs = {
        'left_upper_arm': 'left_shoulder',
        'left_lower_arm': 'left_elbow',
        'left_hand': 'left_wrist',
        'right_upper_arm': 'right_shoulder',
        'right_lower_arm': 'right_elbow',
        'right_hand': 'right_wrist',
        'left_upper_leg': 'left_hip',
        'left_lower_leg': 'left_knee',
        'left_foot': 'left_ankle',
        'right_upper_leg': 'right_hip',
        'right_lower_leg': 'right_knee',
        'right_foot': 'right_ankle',
    }
    for target_name, source_name in joint_specs.items():
        landmark = _single_joint_landmark(target_name, joints_by_name, source_name, frame_width, frame_height)
        if landmark:
            landmarks[target_name] = landmark

    if 'hips' in landmarks and 'shoulders' in landmarks:
        hips = landmarks['hips']['position']
        shoulders = landmarks['shoulders']['position']
        landmarks['spine'] = {
            'name': 'spine',
            'position': {
                'x': float((hips['x'] + shoulders['x']) / 2.0),
                'y': float((hips['y'] + shoulders['y']) / 2.0),
                'z': 0.0,
            },
            'confidence': float((landmarks['hips']['confidence'] + landmarks['shoulders']['confidence']) / 2.0),
            'visible': True,
            'source_joints': ['hips', 'shoulders'],
        }

    return landmarks


_current_video_id = globals().get('CURRENT_VIDEO_ID')
if 'yolo_frame_records' in globals():
    if _current_video_id is not None and globals().get('yolo_source_video_id') != _current_video_id:
        raise RuntimeError('YOLO pose results are from a different input video. Re-run the YOLO Pose fallback cell for the current video.')
    pose_frame_records = yolo_frame_records
    pose_df = yolo_df
    pose_source_name = 'yolo'
elif 'frame_records' in globals():
    if _current_video_id is not None and globals().get('rfdetr_source_video_id', _current_video_id) != _current_video_id:
        raise RuntimeError('RF-DETR pose results are from a different input video. Re-run RF-DETR inference for the current video.')
    pose_frame_records = frame_records
    pose_df = df
    pose_source_name = globals().get('active_pose_source', 'current frame_records')
else:
    raise RuntimeError('No pose frame records found. Re-run pose inference for the current video.')
print(f"motion source={pose_source_name} frames={len(pose_frame_records)}")

motion_frames = []
for frame in pose_frame_records:
    people = []
    for person in frame.get('people', []):
        people.append({
            'person_index': person['person_index'],
            'track_id': person.get('track_id'),
            'detection_confidence': person.get('detection_confidence'),
            'bbox_xyxy': person.get('bbox_xyxy'),
            'keypoints': person.get('keypoints', []),
            'humanoid_landmarks': _build_humanoid_landmarks(person, width, height),
            'coco_keypoints_normalized': {
                item['joint_name']: {
                    'position': _normalize_point(item['x'], item['y'], width, height),
                    'confidence': item['keypoint_confidence'],
                    'visible': item['visible'],
                    'joint_index': item['joint_index'],
                }
                for item in person.get('keypoints', [])
            },
        })
    motion_frames.append({
        'frame_index': frame['frame_index'],
        'time_s': frame['time_s'],
        'people': people,
    })

motion_payload = {
    'schema': 'er_flowscan.rfdetr_keypoint_motion.v1',
    'source_video': str(video_path),
    'source_video_id': dict(globals().get('CURRENT_VIDEO_ID', {'path': str(video_path)})),
    'pose_source': pose_source_name,
    'fps': float(fps / max(FRAME_STRIDE, 1)),
    'frame_width': int(width),
    'frame_height': int(height),
    'coordinate_system': {
        'type': 'normalized_screen_space_2_5d',
        'x': 'right_positive_center_origin',
        'y': 'up_positive_center_origin',
        'z': 'placeholder_zero_from_single_view_2d_pose',
        'unit': 'max(frame_width, frame_height)',
    },
    'retargeting_note': 'Use humanoid_landmarks as IK targets. For real depth, run a 2D-to-3D pose lifter before retargeting.',
    'coco_keypoint_names': COCO_KEYPOINT_NAMES,
    'coco_skeleton': COCO_SKELETON,
    'frames': motion_frames,
}

motion_payload_source_video_id = dict(globals().get('CURRENT_VIDEO_ID', {'path': str(video_path)}))
motion_json_path = OUTPUT_DIR / 'character_motion.json'
motion_json_path.write_text(json.dumps(motion_payload, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"character_motion_json={motion_json_path}")
print(f"frames={len(motion_frames)}")
print(json.dumps(motion_payload['frames'][0] if motion_payload['frames'] else {}, ensure_ascii=False, indent=2)[:2000])

### Three.jsで取り込む最小例

`character_motion.json` の `frames[*].people[*].humanoid_landmarks` を読み、VRMやGLTFキャラクターのIKターゲットに渡します。まずはhips/shoulders/head/hand/footをターゲットにすると動きが確認しやすいです。

In [ ]:
threejs_example_path = OUTPUT_DIR / 'threejs_motion_example.js'
threejs_example_path.write_text("""
// Minimal import example. Replace `ikTargets` with your VRM/GLTF IK target objects.
const motion = await fetch('./character_motion.json').then((res) => res.json());
const ikTargets = {
  hips: null,
  head: null,
  left_hand: null,
  right_hand: null,
  left_foot: null,
  right_foot: null,
};

let startTime = performance.now();
function applyFrame(now) {
  const t = (now - startTime) / 1000;
  const frameIndex = Math.min(Math.floor(t * motion.fps), motion.frames.length - 1);
  const person = motion.frames[frameIndex]?.people?.[0];
  if (person) {
    for (const [name, landmark] of Object.entries(person.humanoid_landmarks)) {
      const target = ikTargets[name];
      if (!target || !landmark.visible) continue;
      const p = landmark.position;
      target.position.set(p.x, p.y, p.z);
    }
  }
  requestAnimationFrame(applyFrame);
}
requestAnimationFrame(applyFrame);
""".strip(), encoding='utf-8')
print(f"threejs_example={threejs_example_path}")

## 10. 横長の診断動画は既定では作らない

以前の `side_by_side_mannequin.mp4` は、元動画の横に別パネルを連結する診断用動画でした。今回の目的は「元動画内に色付きの関節キャラクターを合成した音声付き動画」なので、この横長診断動画は既定では生成しません。最終出力は後段の色付き関節キャラクター合成セルで作る `in_frame_character_rigged.mp4` です。

In [ ]:
# Side-by-side diagnostic output is intentionally disabled by default.
# The target deliverable is the in-frame colored articulated character video.
GENERATE_SIDE_BY_SIDE_DIAGNOSTIC = globals().get('GENERATE_SIDE_BY_SIDE_DIAGNOSTIC', False)
side_raw_path = OUTPUT_DIR / 'side_by_side_mannequin_raw.mp4'
side_final_path = OUTPUT_DIR / 'side_by_side_mannequin.mp4'
written = 0

if GENERATE_SIDE_BY_SIDE_DIAGNOSTIC:
    raise RuntimeError(
        'The old side-by-side mannequin diagnostic is disabled in this notebook version. '
        'Use the colored articulated character cell instead so the composite stays single-width.'
    )

print('Skipping side-by-side diagnostic video. The final output will stay single-width.')
print('Run the colored articulated character cell and the following transcode/display cell for the audio-preserved MP4.')

In [ ]:
# No-op for the disabled side-by-side diagnostic section.
# Audio is preserved later by transcode_with_progress(), which maps the original video audio with -map 1:a:0?.
print('Skipping side-by-side transcode; no wide diagnostic video generated.')

## 11. 旧 in-frame mannequin は無効化

このセクションは以前の簡易3D人形 `in_frame_mannequin.mp4` 用でした。今回の正式出力は、後段の色付き関節キャラクター合成 `in_frame_character_rigged.mp4` です。別キャラクターが混ざらないよう、この旧経路は実行しても動画を生成しません。

In [ ]:
# Disabled legacy in-frame mannequin output.
# Use the colored articulated character cell below instead.
GENERATE_LEGACY_IN_FRAME_MANNEQUIN = globals().get('GENERATE_LEGACY_IN_FRAME_MANNEQUIN', False)
in_frame_raw_path = OUTPUT_DIR / 'in_frame_mannequin_raw.mp4'
in_frame_final_path = OUTPUT_DIR / 'in_frame_mannequin.mp4'
in_frame_written = 0

if GENERATE_LEGACY_IN_FRAME_MANNEQUIN:
    raise RuntimeError(
        'Legacy in-frame mannequin generation is disabled to avoid mixing outputs. '
        'Run the colored articulated character cell below.'
    )

print('Skipping legacy in-frame mannequin. Use the colored articulated character output.')

In [ ]:
# No-op for disabled legacy in-frame mannequin transcode.
print('Skipping legacy in-frame mannequin transcode.')

## 12. 旧 1枚画像 sprite キャラクターは無効化

このセクションは `色付きにけどら.png` を1枚画像として拡大・回転・上下移動する方式でした。関節は個別に動かないため、今回の目的動画には使いません。正式経路は次の色付き関節キャラクター合成です。

In [ ]:
# Disabled single-sprite character output.
# It only rotates/scales a flat image and does not move joints independently.
GENERATE_LEGACY_SPRITE_CHARACTER = globals().get('GENERATE_LEGACY_SPRITE_CHARACTER', False)
character_raw_path = OUTPUT_DIR / 'in_frame_character_sprite_raw.mp4'
character_final_path = OUTPUT_DIR / 'in_frame_character_sprite.mp4'
written = 0

if GENERATE_LEGACY_SPRITE_CHARACTER:
    raise RuntimeError(
        'Legacy single-sprite character generation is disabled for this workflow. '
        'Run the colored articulated character cell below.'
    )

print('Skipping legacy single-sprite character. Use the colored articulated character output.')

## 13. 色付き関節キャラクターを元動画内に合成する

ここからが今回の正式出力です。横長の診断動画や1枚画像spriteではなく、元動画サイズの中に色付きの関節キャラクターを合成します。次のセルで `in_frame_character_rigged_raw.mp4` を生成し、その次のセルで元動画の音声を保持した `in_frame_character_rigged.mp4` に変換します。
このノートブック(v3)では、キーポイント人形を既定でトラッキング対象ダンサーの「左後ろ」に配置し、YOLOセグメンテーション(yolov8x-seg, retina masks)でダンサー本人を高品質に切り抜いて、人形を描画した後にダンサーだけを最前面へ合成し直します。二人が重なった部分は人形がダンサーの後ろに隠れ、現実世界に合成されているような奥行き表現になります。`DANCER_OCCLUSION_ENABLED = False` で従来動作(人形が常に前面)へ、`CHARACTER_RELATIVE_X_OFFSET` を正の値にすれば右側配置へ戻せます。

足滑り対策: 既定の `CHARACTER_PLACEMENT_MODE = 'world_locked'` では、床・壁など背景の特徴点を毎フレーム追跡してカメラ運動をホモグラフィ(射影変換)として推定し、人形の足元を現実世界の床の一点に固定します。人物bboxに追従しないので、ダンサーが動いても人形が床を滑りません。カメラのパン・ズーム・手ブレにも追従します。`CHARACTER_WORLD_LOCK_REATTACH_GAIN`(既定0.02)は、ダンサーが大きく移動した時に人形がゆっくり付いていく強さです。

このノートブック(v4)は、セル5bのMediaPipe 3D姿勢推定があるフレームで高度化描画を行います: (1) 関節ごとの深度(z)で胴体・四肢・頭を並べ替え、手が顔の前にある時は手を手前に、後ろにある時は顔を手前に描画します。(2) 肩と腰の横ズレから「腰のくねり」を曲がった背骨と6点ポリゴンの胴体で表現します(`CHARACTER_TORSO_BEND_GAIN`で誇張率を調整)。MediaPipe結果が無いフレームは従来の2D描画へ自動フォールバックします。


In [ ]:
# Legacy custom rig asset preparation is not needed for the colored articulated renderer below.
# The renderer draws the character directly from pose keypoints.
print('Skipping legacy custom rig asset preparation.')

In [ ]:
# Legacy Retargeter/PuppetRenderer support is disabled for this workflow.
# The next cell uses the pose keypoints directly to draw a colored articulated character.
print('Skipping legacy Retargeter/PuppetRenderer definitions.')

In [ ]:
# --- self-contained colored articulated character render cell
# Replaces the previous rigged character cell.
#
# This version does not depend on PuppetRenderer/Retargeter for the visible
# character. It uses the detected pose keypoints directly, so elbows, knees,
# wrists, ankles, head, and torso move independently instead of holding one
# sprite and shaking it.

import math
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm

def _current_video_identity_for_check():
    if 'CURRENT_VIDEO_ID' in globals():
        return CURRENT_VIDEO_ID
    if 'video_path' in globals() and '_video_identity' in globals():
        return _video_identity(video_path)
    return None


def _assert_pose_source_matches_current(source_video_id, source_name):
    current_video_id = _current_video_identity_for_check()
    if current_video_id is not None and source_video_id is None:
        raise RuntimeError(
            f'{source_name} pose results do not have a source video id. '
            'Re-run the pose inference cell once so the character render can verify it matches the current input video.'
        )
    if current_video_id is not None and source_video_id != current_video_id:
        raise RuntimeError(
            f'{source_name} pose results are from a different video. '
            'Re-run the video input cell and pose inference cells from the top for the current video.'
        )


if 'yolo_frame_records' in globals():
    _assert_pose_source_matches_current(globals().get('yolo_source_video_id'), 'YOLO')
    character_motion_by_frame = {int(frame['frame_index']): frame for frame in yolo_frame_records}
    character_motion_source = 'yolo_frame_records'
elif 'frame_records' in globals():
    _assert_pose_source_matches_current(globals().get('rfdetr_source_video_id'), 'RF-DETR')
    character_motion_by_frame = {int(frame['frame_index']): frame for frame in frame_records}
    character_motion_source = 'frame_records'
elif 'motion_payload' in globals():
    _assert_pose_source_matches_current(
        globals().get('motion_payload_source_video_id', motion_payload.get('source_video_id')),
        'motion_payload',
    )
    character_motion_by_frame = {int(frame['frame_index']): frame for frame in motion_payload['frames']}
    character_motion_source = 'motion_payload'
else:
    raise RuntimeError(
        'Run pose inference first. Expected yolo_frame_records, frame_records, or motion_payload in globals().'
    )
print(f'character motion source={character_motion_source} frames={len(character_motion_by_frame)}')

# MediaPipe 3D姿勢結果(セル5b)があれば利用します。
MP3D_BY_FRAME = globals().get('mp3d_by_frame', None)
if MP3D_BY_FRAME and 'CURRENT_VIDEO_ID' in globals() and globals().get('mp3d_source_video_id') != CURRENT_VIDEO_ID:
    print('警告: MediaPipe 3D結果は別動画のものです。セル5bを再実行してください。今回は2Dキーポイントで描画します。')
    MP3D_BY_FRAME = None
print(f"mediapipe_3d_available={bool(MP3D_BY_FRAME)} frames={len(MP3D_BY_FRAME) if MP3D_BY_FRAME else 0}")

# ============================================================
# Character render settings / キャラクター描画設定
# ============================================================

# 1) Puppet placement / 人形を動画内のどこに置くか
# ------------------------------------------------------------------
# ★ 既定: トラッキングしている人物の「左後ろ」で踊らせる
#     負のXオフセットで人物の左側へ、負のYオフセットで少し奥（上）へ置きます。
#     ダンサー切り抜きオクルージョン(下の 1b)が有効なら、重なった部分は
#     ダンサーの後ろへ自然に隠れるので、多少重なる近さに置くのがおすすめです。
#     右下に戻したい場合は X=0.28 / Y=0.18 程度にします。
#     CHARACTER_PLACEMENT_MODE = 'relative_to_person'
#
# ★ 画面の固定位置に置きたい場合:
#     CHARACTER_PLACEMENT_MODE = 'fixed'
#     CHARACTER_POSITION = 2  # 1=左下, 2=中央下, 3=右下
#     CHARACTER_X_FRACTION = None
#
# ★ 手動で横位置を固定したい場合:
#     CHARACTER_PLACEMENT_MODE = 'fixed'
#     CHARACTER_X_FRACTION = 0.50  # 画面中央
#     CHARACTER_X_FRACTION = 0.82  # 右寄り
#
# 重要: この配置系パラメータは globals().get ではなく直接代入です。編集してセルを再実行すれば必ず反映されます。
# 'world_locked'      = ★推奨。床・壁の特徴点でカメラ運動を推定し、人形を現実の床の一点に固定(足滑りなし)。
#                        初期位置は CHARACTER_RELATIVE_X/Y_OFFSET(人物相対)で決まります。
# 'relative_to_person' = 毎フレーム人物bboxに追従。人物が動くと人形も滑るように動きます。
# 'fixed'              = 画面の固定位置。
CHARACTER_PLACEMENT_MODE = 'world_locked'  # 'world_locked' / 'relative_to_person' / 'fixed'
# 人物とどれだけ離すか。「人物bboxの端」と「人形の近い側の端」の間の隙間で、人物幅に対する割合です。
# (人形の半幅は自動で考慮されるので、0.0 でも人形は人物に重ならず真横にぴったり並びます)
# 0以上: 人物の右横に配置。0.10=人物幅の10%の隙間。
# 負の値: 人物の左横に配置。-0.10=人物幅の10%の隙間。左右対称です。
# 左後ろ配置の既定: 人物左端との間に人物幅10%の隙間。ダンサーが左へ動いた時だけ自然に重なります。
# 既定: 隙間ほぼゼロ(左横にぴったり)。さらに近づけたい時は下の TUCK を上げます。
CHARACTER_RELATIVE_X_OFFSET = -0.01
# 人形の見かけ半幅。人形の高さに対する割合です。上の隙間計算に使います。
# 人形がまだ人物に重なって見える場合は 0.35-0.40 へ上げ、離れすぎる場合は 0.25 へ下げてください。
CHARACTER_BODY_HALF_WIDTH_FRACTION = 0.24
# 人形を人物側へ食い込ませる量。人物幅に対する割合です。オクルージョン有効時は
# 食い込んだ部分がダンサーの後ろに隠れ、「すぐ後ろで踊っている」見た目になります。
# 0.0=食い込みなし(隙間どおり)。0.10-0.25が目安。大きいほどダンサーに近づき、深く重なります。
CHARACTER_X_TUCK_FRACTION = 0.12
# relative_to_person時: 選択人物bboxの下方向へどれだけずらすか。
# 0.18=人物bbox高さの18%ぶん下へ。0.00だと右横に見えやすく、負数で上へ。
# 左「後ろ」らしく、足元をダンサーより人物高さの4%だけ上（画面奥側）へ。
# 床レベル推定が有効な時は「推定した床」からのオフセットになります。-0.03=床より人物高さ3%だけ奥(上)。
CHARACTER_RELATIVE_Y_OFFSET = -0.03

# 床レベル推定 / Floor level tracking
# ダンサーの足首の接地位置(直近ウィンドウの高パーセンタイル=一番低い接地点付近)を追跡して
# 「床がどこか」を推定し、人形の足元をその床に置きます。ダンサーのジャンプやbbox下端の
# 揺れで人形が宙に浮くのを防ぎます。
CHARACTER_FLOOR_LEVEL_ENABLED = globals().get('CHARACTER_FLOOR_LEVEL_ENABLED', True)
# 床推定に使う直近フレーム数。長いほど安定、カメラの上下移動には追従が遅くなります。
CHARACTER_FLOOR_LEVEL_WINDOW_FRAMES = globals().get('CHARACTER_FLOOR_LEVEL_WINDOW_FRAMES', 45)
# 接地点のパーセンタイル。大きいほど「一番低い点=床」寄り。ジャンプの多いダンスでは88-95。
CHARACTER_FLOOR_LEVEL_PERCENTILE = globals().get('CHARACTER_FLOOR_LEVEL_PERCENTILE', 90)
# 床推定に使う足首キーポイントの最低信頼度。
CHARACTER_FLOOR_ANKLE_CONF = globals().get('CHARACTER_FLOOR_ANKLE_CONF', 0.30)
# relative_to_person時: 相対位置をどれだけ画面端で制限するか。
# 以前は画面内に強く収めていましたが、ダンスでは手足や一部関節が画面外へ出る方が自然です。
# ここは最小限の暴走防止用です。通常は 0.00 のままで、下の PARTIAL_OFFSCREEN で画面外を許可します。
CHARACTER_RELATIVE_MARGIN_FRACTION = 0.00
# 人形の一部が画面外へ出ることを許可する量。人形高さに対する割合です。
# 0.65なら、アンカーが画面端をかなり越えても相対位置を優先します。
CHARACTER_ALLOW_PARTIAL_OFFSCREEN_FRACTION = 0.65
# fixed時だけ使用。1=左下, 2=中央下, 3=右下。
CHARACTER_POSITION = 2
# fixed時の手動横位置。Noneなら CHARACTER_POSITION を使用。0.50=中央, 0.82=右寄り。
CHARACTER_X_FRACTION = None
# fixed時の足元・人形下端の縦位置。大きくすると下へ移動します。目安: 0.86-0.94。
CHARACTER_Y_FRACTION = 0.90
# 人形全体の高さ。動画の高さに対する割合です。大きくすると人形が大きくなります。目安: 0.25-0.36。既定は以前より約20%小さめ。
# 後方配置では少し小さめにすると遠近感が出ます。
CHARACTER_HEIGHT_FRACTION = 0.27
# 人形全体の不透明度。1.0で完全に不透明、0.5で半透明。
CHARACTER_OPACITY = 0.96  # ここを直接変更: 1.0=完全不透明

# 1b) Dancer occlusion / ダンサーを人形より前面（最上位レイヤー）に表示
# ------------------------------------------------------------------
# True: セグメンテーションでトラッキング対象のダンサー本人を切り抜き、人形を描いた後に
#       ダンサーだけを最前面へ合成し直します。ダンサーと人形が重なった時、人形が
#       ダンサーの後ろに隠れて見えるので、人形が現実世界に自然に合成されているように感じられます。
# False: 従来通り、人形を常にダンサーの上に描画します（重なると人形が前に出ます）。
DANCER_OCCLUSION_ENABLED = globals().get('DANCER_OCCLUSION_ENABLED', True)
# ダンサー切り抜きに使うYOLOセグメンテーションモデル。速さ優先: yolov8n-seg.pt / 精度優先: yolov8x-seg.pt
DANCER_SEG_MODEL_NAME = globals().get('DANCER_SEG_MODEL_NAME', 'yolov8x-seg.pt')
DANCER_SEG_CONF_THRESHOLD = globals().get('DANCER_SEG_CONF_THRESHOLD', 0.25)
DANCER_SEG_IMGSZ = globals().get('DANCER_SEG_IMGSZ', 960)
# True: マスクを入力解像度で生成(retina masks)。輪郭が滑らかになり、指先・髪などの境界品質が上がります。
DANCER_SEG_RETINA_MASKS = globals().get('DANCER_SEG_RETINA_MASKS', True)
# 選択中のtrack人物のbboxと、セグメンテーション結果の各インスタンスbboxのIoUで同一人物かを判定します。
# これ未満のIoUは「別人/誤検出」とみなし、その人物は切り抜きません（=そのフレームは人形が前面のまま）。
DANCER_SEG_MATCH_IOU_MIN = globals().get('DANCER_SEG_MATCH_IOU_MIN', 0.10)
# 切り抜きマスクの輪郭をぼかす量(px)。大きいほど境界が自然になりますが、輪郭付近が少し透けます。
DANCER_SEG_MASK_FEATHER_PX = globals().get('DANCER_SEG_MASK_FEATHER_PX', 2)
# マスクを少し膨張させる量(px)。輪郭のすぐ外側に人形の線が透けて見える問題を抑えます。
DANCER_SEG_MASK_DILATE_PX = globals().get('DANCER_SEG_MASK_DILATE_PX', 1)
# 高速化: セグメンテーションを何フレームおきに実行するか。1=毎フレーム(推奨)、
# 2以上は実行しないフレームで直前のマスクを再利用します（速い動きでは輪郭が少し遅れます）。
DANCER_SEG_EVERY_N_FRAMES = globals().get('DANCER_SEG_EVERY_N_FRAMES', 1)

dancer_seg_model = None
if DANCER_OCCLUSION_ENABLED:
    if 'YOLO' not in globals():
        from ultralytics import YOLO
    dancer_seg_model = YOLO(DANCER_SEG_MODEL_NAME)
    print(
        f'dancer occlusion enabled: seg_model={DANCER_SEG_MODEL_NAME} conf={DANCER_SEG_CONF_THRESHOLD} '
        f'imgsz={DANCER_SEG_IMGSZ} retina_masks={DANCER_SEG_RETINA_MASKS}'
    )
else:
    print('dancer occlusion disabled: puppet will always render above the dancer.')

# 1c) World-locked placement / 床・壁の特徴点で人形を現実空間に固定
# ------------------------------------------------------------------
# world_locked は足滑り対策の決定版です。毎フレーム、床・壁など背景全体の特徴点を追跡して
# カメラの動き(パン・チルト・ズーム・手ブレ)をホモグラフィ(射影変換)として推定し、
# 人形の足元アンカーを「現実世界の床の一点」に貼り付けたまま画面座標へ変換し続けます。
# 人物bboxには追従しないため、ダンサーが動いても人形の足元は滑りません。
# (完全な3D復元ではなく平面ホモグラフィ近似ですが、床上の一点の固定にはこれで十分です)
#
# 世界固定アンカーを、人物相対の目標位置へゆっくり引き戻す強さ(1フレームあたりの割合)。
# 0.0 = 完全に床へ固定(ダンサーが遠くへ移動すると人形が置いていかれます)。
# 0.01-0.03 = ダンサーが画面内を大きく移動する動画向け。ごくゆっくり付いていき、滑りはほぼ見えません。
CHARACTER_WORLD_LOCK_REATTACH_GAIN = globals().get('CHARACTER_WORLD_LOCK_REATTACH_GAIN', 0.02)
# カメラのズームに合わせて人形の大きさも変える。True推奨(ズーム時に人形だけ大きさが変わらない違和感を防ぐ)。
CHARACTER_WORLD_SCALE_WITH_CAMERA = globals().get('CHARACTER_WORLD_SCALE_WITH_CAMERA', True)
# 1フレームで人形サイズが変われる最大割合(ズーム追従の暴走防止)。
CHARACTER_WORLD_SCALE_MAX_STEP = globals().get('CHARACTER_WORLD_SCALE_MAX_STEP', 0.05)
# 背景特徴点解析の画像幅。大きいほど精密ですが重くなります。480-960が目安。
CHARACTER_WORLD_MOTION_ANALYSIS_WIDTH = globals().get('CHARACTER_WORLD_MOTION_ANALYSIS_WIDTH', 640)
# 追跡する特徴点の最大数。多いほどロバストで少し重い。
CHARACTER_WORLD_MOTION_MAX_CORNERS = globals().get('CHARACTER_WORLD_MOTION_MAX_CORNERS', 240)
# ホモグラフィ採用に必要な最小インライア数。足りないフレームは前フレームの画面位置を維持します。
CHARACTER_WORLD_MOTION_MIN_INLIERS = globals().get('CHARACTER_WORLD_MOTION_MIN_INLIERS', 12)
# RANSAC再投影誤差しきい値(px, フル解像度換算)。小さいほど厳密。
CHARACTER_WORLD_MOTION_RANSAC_REPROJ = globals().get('CHARACTER_WORLD_MOTION_RANSAC_REPROJ', 3.0)


# 1d) MediaPipe 3D描画 / 深度ソートと腰のくねり (セル5b実行時に有効)
# ------------------------------------------------------------------
# セル5bの mp3d_by_frame があるフレームでは、キーポイントをMediaPipe 3Dへ置き換えます。
CHARACTER_USE_MP3D = globals().get('CHARACTER_USE_MP3D', True)
# 関節ごとの深度(z)で描画順を並べ替えます。手が顔・胴体の前か後ろかを実際の奥行きで判定します。
CHARACTER_DEPTH_SORT_ENABLED = globals().get('CHARACTER_DEPTH_SORT_ENABLED', True)
# 腰のくねり(ウエストの曲がり)の誇張率。1.0=推定そのまま。1.3-1.6で少し大げさに。0.0で無効(直線胴体)。
CHARACTER_TORSO_BEND_GAIN = globals().get('CHARACTER_TORSO_BEND_GAIN', 1.35)
# 腰の横揺れ(ヒップスウェイ)の誇張率。肩中心に対する腰の横ズレを増幅します(膝は60%連動、足首は固定)。
# 1.0=推定そのまま。1.4-1.8で「腰をくねらせている」動きがはっきり見えます。
CHARACTER_HIP_SWAY_GAIN = globals().get('CHARACTER_HIP_SWAY_GAIN', 1.6)
# 深度(z)の時間平滑化。MediaPipeのzはフレーム間ノイズが大きく、描画の前後関係が
# ちらつく原因になるため平滑化します。大きいほど安定、前後の切替はゆっくりに。0.0で無効。
CHARACTER_Z_SMOOTHING = globals().get('CHARACTER_Z_SMOOTHING', 0.6)


# 足滑り対策 / Foot lock
# Trueにすると、合成人形の足元アンカーを安定させます。
# 人物bboxの下端が検出揺れで上下しても、人形全体が床を滑るように動くのを抑えます。
CHARACTER_FOOT_LOCK_ENABLED = True
# 横方向アンカーの平滑化。大きいほど左右の急な滑りを抑えます。0.80-0.94が目安。
CHARACTER_ANCHOR_X_SMOOTHING = 0.88
# 足元の縦方向アンカーの平滑化。大きいほど床位置が安定します。0.90-0.98が目安。
CHARACTER_ANCHOR_Y_SMOOTHING = 0.94
# 横方向アンカーが1フレームで動ける最大量。人形高さに対する割合です。
CHARACTER_ANCHOR_X_MAX_STEP = 0.10
# 足元が上方向へ動ける最大量。小さいほど床から浮き上がりにくくなります。
CHARACTER_ANCHOR_Y_UP_MAX_STEP = 0.018
# 足元が下方向へ動ける最大量。接地位置が本当に下がった時だけゆっくり追従します。
CHARACTER_ANCHOR_Y_DOWN_MAX_STEP = 0.045
# カメラがパン・ズーム・手ブレした時に、床/背景の特徴点移動で足元アンカーを補正します。
# True推奨。人物bboxだけではなく床側の動きも見るので、現実世界に貼り付いている感じが出やすくなります。
CHARACTER_FLOOR_MOTION_ENABLED = True
# 床特徴点を見る範囲。0.45なら画面の下45%を主に床/地面として見ます。
CHARACTER_FLOOR_ROI_BOTTOM_FRACTION = 0.45
# 床特徴点の移動を足元アンカーへ反映する強さ。0.0=無効、1.0=そのまま反映。
CHARACTER_FLOOR_MOTION_GAIN = 0.85
# 1フレームで床補正できる最大量。大きすぎる特徴点誤検出を弾きます。
CHARACTER_FLOOR_MOTION_MAX_STEP_FRACTION = 0.08
# 高速化: 床特徴点追跡を何フレームおきに実行するか。1=毎フレーム、3=3フレームに1回。
CHARACTER_FLOOR_MOTION_EVERY_N_FRAMES = globals().get('CHARACTER_FLOOR_MOTION_EVERY_N_FRAMES', 3)
# 高速化: 床特徴点解析の画像幅。小さいほど高速。0なら元解像度。
CHARACTER_FLOOR_MOTION_ANALYSIS_WIDTH = globals().get('CHARACTER_FLOOR_MOTION_ANALYSIS_WIDTH', 480)
# 高速化: 特徴点数。少ないほど高速。
CHARACTER_FLOOR_MOTION_MAX_CORNERS = globals().get('CHARACTER_FLOOR_MOTION_MAX_CORNERS', 45)

# 2) Target person selection / どの人物の動きを使うか
#    推奨: YOLO Pose fallbackセルを実行し、表示される track_ids=[1, 2, ...] から選びます。
#    解析動画にも "track 2" のようなラベルが出ます。
#    例: track 2 の人物を使いたい場合は CHARACTER_SOURCE_TRACK_ID = 2。
#    None の場合は下の CHARACTER_SOURCE_PERSON で選びます。
CHARACTER_SOURCE_TRACK_ID = globals().get('CHARACTER_SOURCE_TRACK_ID', None)
# track IDを指定しない時の選び方。
#   'largest'            = bboxが一番大きい人物。メイン人物向け。
#   'highest_confidence' = 検出信頼度が一番高い人物。
#   'rightmost'          = 一番右の人物。
#   'leftmost'           = 一番左の人物。
#   0, 1, 2...           = フレーム内の検出順。track_idより不安定です。
CHARACTER_SOURCE_PERSON = globals().get('CHARACTER_SOURCE_PERSON', 'largest')
# 使うキーポイントの最低信頼度。下げると関節が欠けにくいがノイズが増えます。上げると安定するが欠けやすいです。
KEYPOINT_CONF_THRESHOLD = globals().get('KEYPOINT_CONF_THRESHOLD', 0.25)

# 3) Body drawing style / 体・関節の描画スタイル
# 手足や関節の線の太さ。大きくすると太い人形になります。
CHARACTER_LINE_SCALE = globals().get('CHARACTER_LINE_SCALE', 1.0)
# 頭サイズの基礎倍率。画像頭にも効きます。頭だけ大きい/小さい時に調整します。
CHARACTER_HEAD_SCALE = globals().get('CHARACTER_HEAD_SCALE', 1.0)
# 関節位置の平滑化。大きくすると滑らか・安定、小さくすると反応が速い。目安: 0.25-0.50。
CHARACTER_SMOOTHING = globals().get('CHARACTER_SMOOTHING', 0.35)
# Trueにすると、選ばれた元人物のbboxを動画上に表示します。対象人物の確認用。
CHARACTER_DRAW_SOURCE_DEBUG = globals().get('CHARACTER_DRAW_SOURCE_DEBUG', False)
# 手・腕を顔画像より前面に描画します。顔の前で手を動かす動作では、手が顔よりカメラ側に
# あることがほとんどのため True が自然です。2Dキーポイントには奥行きがなく実際の前後関係は
# 判定できないので、「手が手前」を既定とします。Falseで従来どおり顔画像が常に最前面。
CHARACTER_HANDS_OVER_FACE = globals().get('CHARACTER_HANDS_OVER_FACE', True)

# 4) Head image settings / 頭画像の設定
# Colabでは /content、Google Drive、またはアップロードからこの画像を探します。
CHARACTER_HEAD_IMAGE_FILENAME = globals().get(
    'CHARACTER_HEAD_IMAGE_FILENAME',
    '3f9822740a8792a08f99b77b2fb78940~tplv-tiktokx-cropcenter_1080_1080.jpeg',
)
# 画像のパスを直接指定したい場合に使います。通常は None でOK。
CHARACTER_HEAD_IMAGE_PATH = globals().get('CHARACTER_HEAD_IMAGE_PATH', None)
# Driveに頭画像を置く場合の保存先/探索先。
CHARACTER_HEAD_IMAGE_DRIVE_DIR = Path(globals().get('CHARACTER_HEAD_IMAGE_DRIVE_DIR', '/content/drive/MyDrive/dancing_character_assets'))
# 'square' = 画像をそのまま四角く使う。顔写真/看板画像はこちらが推奨。
# 'neon_alpha' = 暗い背景を透過し、明るいネオン線だけを残す。
CHARACTER_HEAD_IMAGE_ALPHA_MODE = globals().get('CHARACTER_HEAD_IMAGE_ALPHA_MODE', 'square')
# neon_alpha の時だけ使用。大きくすると暗い部分がより消えます。
CHARACTER_HEAD_IMAGE_DARK_THRESHOLD = globals().get('CHARACTER_HEAD_IMAGE_DARK_THRESHOLD', 34)
# 頭画像の大きさ。大きくすると首や肩を隠しやすいので、気になる時は 1.4-1.6 へ下げます。
CHARACTER_HEAD_IMAGE_SCALE = globals().get('CHARACTER_HEAD_IMAGE_SCALE', 1.72)
# 画像内のどこを首に接続するか。1.04は画像下端より少し下を首に合わせ、首・肩を隠しにくくします。
# 小さくすると画像が下がって首にかぶり、大きくすると画像が上がります。
CHARACTER_HEAD_IMAGE_ANCHOR_Y = globals().get('CHARACTER_HEAD_IMAGE_ANCHOR_Y', 1.04)
# 'neck' = 首/肩の中点に画像下部を接続。推奨。
# 'face_center' = 検出された顔中心に追従。顔検出が跳ねる動画では不安定になりやすいです。
CHARACTER_HEAD_IMAGE_ATTACH_MODE = globals().get('CHARACTER_HEAD_IMAGE_ATTACH_MODE', 'neck')
# 首基準からの上下補正。負の値で頭画像を上へ移動。目安: -0.03 から -0.08。
CHARACTER_HEAD_IMAGE_NECK_OFFSET_FRACTION = globals().get('CHARACTER_HEAD_IMAGE_NECK_OFFSET_FRACTION', -0.04)
# 頭画像の不透明度。
CHARACTER_HEAD_IMAGE_OPACITY = globals().get('CHARACTER_HEAD_IMAGE_OPACITY', 1.0)
# 頭画像の回転モード。
#   'face_roll' = 目/耳/鼻の傾きから頭のロール角を推定して、首の接続点を支点に回転。推奨。
#   'shoulders' = 肩の傾きだけを弱く使う。
#   'off'       = 常に水平。
CHARACTER_HEAD_IMAGE_ROTATION_MODE = globals().get('CHARACTER_HEAD_IMAGE_ROTATION_MODE', 'face_roll')
# 旧設定との互換用。Trueなら shoulders と同等に扱います。
CHARACTER_HEAD_IMAGE_ROTATE_WITH_SHOULDERS = globals().get('CHARACTER_HEAD_IMAGE_ROTATE_WITH_SHOULDERS', False)
# 回転を有効化した時の最大角度。ぐるぐる回らないための上限です。
CHARACTER_HEAD_IMAGE_MAX_ROTATION_DEG = globals().get('CHARACTER_HEAD_IMAGE_MAX_ROTATION_DEG', 14.0)
# 1フレームで変化できる頭角度の上限。小さいほど急回転しません。
CHARACTER_HEAD_IMAGE_MAX_ROTATION_STEP_DEG = globals().get('CHARACTER_HEAD_IMAGE_MAX_ROTATION_STEP_DEG', 4.0)
# 頭角度の平滑化。大きいほど滑らかですが少し遅れます。
CHARACTER_HEAD_IMAGE_ROTATION_SMOOTHING = globals().get('CHARACTER_HEAD_IMAGE_ROTATION_SMOOTHING', 0.72)
# 振り返り/横向きの簡易検知を、軽い回転補正として足す最大角度。2D推定なので控えめにします。
CHARACTER_HEAD_TURN_ROTATION_DEG = globals().get('CHARACTER_HEAD_TURN_ROTATION_DEG', 4.0)

# 5) Stabilizers / 不自然なジャンプを抑える設定
# キーポイント欠落・誤検出・人物入れ替わりで、体や頭が急に巨大化/縮小/ワープするのを抑えます。
# 大きくすると安定しますが、動きに少し遅れが出ます。
CHARACTER_SCALE_SMOOTHING = globals().get('CHARACTER_SCALE_SMOOTHING', 0.88)
# 1フレームあたりの体スケール変化上限。小さくするとサイズ跳ねが減ります。目安: 0.03-0.08。
CHARACTER_SCALE_MAX_STEP = globals().get('CHARACTER_SCALE_MAX_STEP', 0.06)
# 元人物の身長推定の平滑化。大きいほど身長ベースの拡大縮小が安定します。
CHARACTER_SOURCE_HEIGHT_SMOOTHING = globals().get('CHARACTER_SOURCE_HEIGHT_SMOOTHING', 0.90)
# 元人物の身長推定が1フレームで変われる最大割合。
CHARACTER_SOURCE_HEIGHT_MAX_STEP = globals().get('CHARACTER_SOURCE_HEIGHT_MAX_STEP', 0.08)
# 元人物の中心/足元位置の平滑化。大きいほど人形が横・上下に跳びにくくなります。
CHARACTER_SOURCE_POSITION_SMOOTHING = globals().get('CHARACTER_SOURCE_POSITION_SMOOTHING', 0.78)
# 元人物の中心/足元位置が1フレームで動ける最大割合。
CHARACTER_SOURCE_POSITION_MAX_STEP = globals().get('CHARACTER_SOURCE_POSITION_MAX_STEP', 0.12)
# track ID未指定時に、前フレームのbboxから同じ人物を追うかどうか。
CHARACTER_TRACK_PERSON = globals().get('CHARACTER_TRACK_PERSON', True)
# 前フレームbboxとの重なりをどれだけ重視するか。大きいほど人物入れ替わりを避けます。
CHARACTER_TRACK_IOU_WEIGHT = globals().get('CHARACTER_TRACK_IOU_WEIGHT', 3.0)
# 前フレーム中心からの距離ペナルティ。大きいほど遠くの人物へ切り替わりにくくなります。
CHARACTER_TRACK_CENTER_WEIGHT = globals().get('CHARACTER_TRACK_CENTER_WEIGHT', 1.0)
# 頭サイズは顔キーポイントの横幅ではなく、体の高さ比で固定します。
CHARACTER_HEAD_RADIUS_FRACTION = globals().get('CHARACTER_HEAD_RADIUS_FRACTION', 0.055)
# 頭半径の下限/上限。頭が急に大きくなる・小さくなる問題を抑えます。
CHARACTER_HEAD_RADIUS_MIN_FRACTION = globals().get('CHARACTER_HEAD_RADIUS_MIN_FRACTION', 0.045)
CHARACTER_HEAD_RADIUS_MAX_FRACTION = globals().get('CHARACTER_HEAD_RADIUS_MAX_FRACTION', 0.070)
# キーポイントや指定track IDが一瞬消えた時に、前フレームの完成済み人形を保持する最大フレーム数。
# 重要: 古い人物データで再計算はせず、完成済み画像を短く保持するだけです。
CHARACTER_MAX_HOLD_FRAMES = globals().get('CHARACTER_MAX_HOLD_FRAMES', 6)
def _character_bbox_center_x(person):
    bbox = person.get('bbox_xyxy') if person else None
    if not bbox:
        return None
    return float((bbox[0] + bbox[2]) / 2.0)


def _character_bbox_area(person):
    bbox = person.get('bbox_xyxy') if person else None
    if not bbox or len(bbox) < 4:
        return 0.0
    return max(0.0, float(bbox[2]) - float(bbox[0])) * max(0.0, float(bbox[3]) - float(bbox[1]))


def _select_character_motion_person(frame_motion, selector='largest', track_id=None):
    people = frame_motion.get('people', []) if frame_motion else []
    if not people:
        return None
    if track_id is not None:
        wanted_track_id = int(track_id)
        matched = next((person for person in people if person.get('track_id') == wanted_track_id), None)
        if matched is not None:
            return matched
        return None
    if isinstance(selector, int):
        return next((person for person in people if person.get('person_index') == selector), people[0])
    if isinstance(selector, str) and selector.isdigit():
        wanted = int(selector)
        return next((person for person in people if person.get('person_index') == wanted), people[0])
    if selector == 'largest':
        return max(people, key=_character_bbox_area)
    if selector == 'highest_confidence':
        return max(people, key=lambda p: float(p.get('detection_confidence') or 0.0))
    if selector == 'rightmost':
        with_centers = [(p, _character_bbox_center_x(p)) for p in people]
        with_centers = [(p, c) for p, c in with_centers if c is not None]
        if with_centers:
            return max(with_centers, key=lambda item: item[1])[0]
    if selector == 'leftmost':
        with_centers = [(p, _character_bbox_center_x(p)) for p in people]
        with_centers = [(p, c) for p, c in with_centers if c is not None]
        if with_centers:
            return min(with_centers, key=lambda item: item[1])[0]
    return people[0]


def _bbox_center_xy(person):
    bbox = person.get('bbox_xyxy') if person else None
    if not bbox or len(bbox) < 4:
        return None
    return np.array([(float(bbox[0]) + float(bbox[2])) * 0.5, (float(bbox[1]) + float(bbox[3])) * 0.5], dtype=np.float32)


def _bbox_iou(a, b):
    if not a or not b or len(a) < 4 or len(b) < 4:
        return 0.0
    ax1, ay1, ax2, ay2 = [float(v) for v in a[:4]]
    bx1, by1, bx2, by2 = [float(v) for v in b[:4]]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = area_a + area_b - inter
    return float(inter / denom) if denom > 0 else 0.0



def _dancer_mask_to_frame_alpha(mask_small, frame_w, frame_h, dilate_px=0, feather_px=0):
    # セグメンテーションマスクを元動画の解像度へ拡大し、輪郭を少し膨張・ぼかして自然な重なりにします。
    mask_u8 = (np.clip(mask_small, 0, 1) * 255).astype(np.uint8)
    if mask_u8.shape[:2] != (frame_h, frame_w):
        mask_u8 = cv2.resize(mask_u8, (frame_w, frame_h), interpolation=cv2.INTER_LINEAR)
    if dilate_px > 0:
        k = int(dilate_px) * 2 + 1
        mask_u8 = cv2.dilate(mask_u8, np.ones((k, k), np.uint8), iterations=1)
    if feather_px > 0:
        mask_u8 = cv2.GaussianBlur(mask_u8, (0, 0), float(feather_px))
    return (mask_u8.astype(np.float32) / 255.0)[:, :, None]


def _select_dancer_mask_for_person(seg_result, person_bbox_xyxy, frame_w, frame_h, iou_min):
    # セグメンテーション結果の中から、キーポイント人形の元になっている人物(track)と
    # 同一人物と思われるインスタンス(bboxのIoUが最大)だけを選び、その切り抜きマスクを返します。
    if seg_result is None or person_bbox_xyxy is None:
        return None
    boxes = getattr(seg_result, 'boxes', None)
    masks = getattr(seg_result, 'masks', None)
    if boxes is None or masks is None or getattr(masks, 'data', None) is None or len(masks.data) == 0:
        return None
    xyxy = boxes.xyxy
    if hasattr(xyxy, 'detach'):
        xyxy = xyxy.detach().cpu().numpy()
    xyxy = np.asarray(xyxy, dtype=np.float32)
    mask_data = masks.data
    if hasattr(mask_data, 'detach'):
        mask_data = mask_data.detach().cpu().numpy()
    mask_data = np.asarray(mask_data, dtype=np.float32)
    best_idx, best_iou = None, 0.0
    for i in range(len(xyxy)):
        iou = _bbox_iou(person_bbox_xyxy, xyxy[i].tolist())
        if iou > best_iou:
            best_iou, best_idx = iou, i
    if best_idx is None or best_iou < float(iou_min):
        return None
    return _dancer_mask_to_frame_alpha(
        mask_data[best_idx], frame_w, frame_h, DANCER_SEG_MASK_DILATE_PX, DANCER_SEG_MASK_FEATHER_PX
    )
def _kp_dict_from_mp3d(record):
    """MediaPipe 3Dランドマークを既存パイプラインと同じCOCO名の辞書へ変換します。zは別辞書で返します。"""
    kp, zm = {}, {}
    for name, value in (record.get('landmarks') or {}).items():
        x, y, z, vis = [float(v) for v in value[:4]]
        if vis < float(KEYPOINT_CONF_THRESHOLD):
            continue
        kp[str(name)] = [x, y, vis]
        zm[str(name)] = z
    return kp, zm


def _z_of(z_map, *names, default=0.0):
    if not z_map:
        return float(default)
    vals = [float(z_map[n]) for n in names if n in z_map]
    if not vals:
        return float(default)
    return float(sum(vals) / len(vals))


def _smooth_z_map(current, previous):
    """関節深度の時間平滑化。描画順(前後関係)のフレーム間ちらつきを抑えます。"""
    if not current:
        return current
    s = float(CHARACTER_Z_SMOOTHING)
    if not previous or s <= 0:
        return dict(current)
    out = {}
    for name, z in current.items():
        pz = previous.get(name)
        out[name] = float(z) if pz is None else float(pz) * s + float(z) * (1.0 - s)
    return out




def _select_tracked_character_person(frame_motion, selector='largest', previous_person=None, track_id=None):
    people = frame_motion.get('people', []) if frame_motion else []
    if not people:
        return None
    if track_id is not None:
        # Track ID指定時は、指定ID以外へは絶対に切り替えない。
        # IDが一時的に消えた場合は None を返し、描画ループ側で完成済みオーバーレイを短時間だけ保持する。
        return _select_character_motion_person(frame_motion, selector, track_id=track_id)
    if not CHARACTER_TRACK_PERSON or previous_person is None:
        return _select_character_motion_person(frame_motion, selector)
    prev_bbox = previous_person.get('bbox_xyxy')
    prev_center = _bbox_center_xy(previous_person)
    if prev_bbox is None or prev_center is None:
        return _select_character_motion_person(frame_motion, selector)
    prev_area = max(1.0, _character_bbox_area(previous_person))
    prev_scale = math.sqrt(prev_area)
    best_person = None
    best_score = -1e9
    for candidate in people:
        iou = _bbox_iou(prev_bbox, candidate.get('bbox_xyxy'))
        center = _bbox_center_xy(candidate)
        center_penalty = 0.0
        if center is not None:
            center_penalty = float(np.linalg.norm(center - prev_center)) / max(1.0, prev_scale)
        same_index_bonus = 0.15 if candidate.get('person_index') == previous_person.get('person_index') else 0.0
        score = float(CHARACTER_TRACK_IOU_WEIGHT) * iou - float(CHARACTER_TRACK_CENTER_WEIGHT) * center_penalty + same_index_bonus
        if score > best_score:
            best_score = score
            best_person = candidate
    return best_person if best_person is not None else _select_character_motion_person(frame_motion, selector)


def _fixed_character_anchor(frame_width, frame_height):
    if CHARACTER_X_FRACTION is not None:
        x_fraction = CHARACTER_X_FRACTION
    else:
        x_fraction = {1: 0.18, 2: 0.50, 3: 0.82}.get(int(CHARACTER_POSITION), 0.82)
    return int(frame_width * x_fraction), int(frame_height * CHARACTER_Y_FRACTION)


FLOOR_LEVEL_STATE = {'samples': []}


def _person_floor_candidate(person):
    """このフレームでの接地候補y(足首の低い方)。足首が取れなければbbox下端。"""
    if not person:
        return None
    best_y = None
    for item in person.get('keypoints', []) or []:
        name = item.get('joint_name') or item.get('name')
        if name not in ('left_ankle', 'right_ankle'):
            continue
        conf = float(item.get('keypoint_confidence', item.get('confidence', 0.0)) or 0.0)
        if conf < float(CHARACTER_FLOOR_ANKLE_CONF):
            continue
        y = float(item.get('y', 0.0))
        if best_y is None or y > best_y:
            best_y = y
    if best_y is not None:
        return best_y
    bbox = person.get('bbox_xyxy')
    if bbox and len(bbox) >= 4:
        return float(bbox[3])
    return None


def _update_floor_level(person):
    candidate = _person_floor_candidate(person)
    if candidate is None:
        return
    samples = FLOOR_LEVEL_STATE['samples']
    samples.append(float(candidate))
    max_len = max(5, int(CHARACTER_FLOOR_LEVEL_WINDOW_FRAMES))
    if len(samples) > max_len:
        del samples[:len(samples) - max_len]


def _estimated_floor_y():
    """推定した床のy座標(画面座標)。サンプル不足ならNone。"""
    samples = FLOOR_LEVEL_STATE['samples']
    if len(samples) < 5:
        return None
    return float(np.percentile(np.asarray(samples, dtype=np.float32), float(CHARACTER_FLOOR_LEVEL_PERCENTILE)))



def _character_anchor_for_person(frame_width, frame_height, person, base_height):
    # world_locked もこの関数を初期位置・引き戻し目標として使うため、人物相対計算を通します。
    if CHARACTER_PLACEMENT_MODE not in ('relative_to_person', 'world_locked') or person is None:
        return _fixed_character_anchor(frame_width, frame_height)
    bbox = person.get('bbox_xyxy')
    if not bbox or len(bbox) < 4:
        return _fixed_character_anchor(frame_width, frame_height)
    x1, y1, x2, y2 = [float(v) for v in bbox[:4]]
    bbox_w = max(1.0, x2 - x1)
    bbox_h = max(1.0, y2 - y1)
    _x_offset = float(CHARACTER_RELATIVE_X_OFFSET)
    # アンカーは人形の中心なので、人形の半幅ぶん余分に離します。
    # これで X_OFFSET は「人物bbox端と人形の端の隙間」になり、0.0でも人形は人物に重なりません。
    _half_w = float(CHARACTER_BODY_HALF_WIDTH_FRACTION) * float(base_height)
    _tuck = bbox_w * float(CHARACTER_X_TUCK_FRACTION)
    if _x_offset >= 0:
        raw_x = x2 + bbox_w * _x_offset + _half_w - _tuck  # 右横: 人物右端 + 隙間 + 半幅 - 食い込み
    else:
        raw_x = x1 + bbox_w * _x_offset - _half_w + _tuck  # 左横: 人物左端 - 隙間 - 半幅 + 食い込み
    _floor_y = _estimated_floor_y() if CHARACTER_FLOOR_LEVEL_ENABLED else None
    if _floor_y is not None:
        # 床基準: ジャンプやbbox揺れの影響を受けない「推定した床」に足元を置きます。
        raw_bottom_y = _floor_y + bbox_h * float(CHARACTER_RELATIVE_Y_OFFSET)
    else:
        raw_bottom_y = y2 + bbox_h * float(CHARACTER_RELATIVE_Y_OFFSET)
    margin_x = frame_width * float(CHARACTER_RELATIVE_MARGIN_FRACTION)
    margin_y = frame_height * float(CHARACTER_RELATIVE_MARGIN_FRACTION)
    offscreen = base_height * float(CHARACTER_ALLOW_PARTIAL_OFFSCREEN_FRACTION)
    # 相対位置を優先するため、画面端で人形を強く止めません。アンカーだけ暴走防止範囲に収めます。
    min_x = -offscreen + margin_x
    max_x = frame_width + offscreen - margin_x
    min_bottom_y = base_height - offscreen + margin_y
    max_bottom_y = frame_height + offscreen * 0.55 - margin_y
    return int(np.clip(raw_x, min_x, max_x)), int(np.clip(raw_bottom_y, min_bottom_y, max_bottom_y))


def _alpha_blend_bgra(frame_bgr, overlay_bgra, opacity=1.0):
    alpha = (overlay_bgra[:, :, 3:4].astype(np.float32) / 255.0) * float(opacity)
    src = overlay_bgra[:, :, :3].astype(np.float32)
    dst = frame_bgr.astype(np.float32)
    return np.clip(src * alpha + dst * (1.0 - alpha), 0, 255).astype(np.uint8)


def _keypoints_dict_from_person(person):
    if not person:
        return {}
    result = {}
    for item in person.get('keypoints', []):
        name = item.get('joint_name') or item.get('name') or item.get('label')
        if not name:
            continue
        conf = float(item.get('keypoint_confidence', item.get('confidence', item.get('score', 0.0))))
        x = float(item.get('x', np.nan))
        y = float(item.get('y', np.nan))
        if math.isnan(conf) or math.isnan(x) or math.isnan(y):
            continue
        if conf < KEYPOINT_CONF_THRESHOLD:
            continue
        result[str(name)] = [x, y, conf]
    return result


def _first_kp(kp, *names):
    for name in names:
        if name in kp:
            return kp[name]
    return None


def _kp_xy(kp, *names):
    item = _first_kp(kp, *names)
    if item is None:
        return None
    return np.array([float(item[0]), float(item[1])], dtype=np.float32)


def _pose_bounds(kp_dict, person):
    points = []
    for value in kp_dict.values():
        points.append([float(value[0]), float(value[1])])
    if len(points) >= 3:
        arr = np.array(points, dtype=np.float32)
        x1, y1 = arr.min(axis=0)
        x2, y2 = arr.max(axis=0)
        pad_x = max(12.0, (x2 - x1) * 0.12)
        pad_y = max(12.0, (y2 - y1) * 0.10)
        return [x1 - pad_x, y1 - pad_y, x2 + pad_x, y2 + pad_y]
    bbox = person.get('bbox_xyxy') if person else None
    if bbox and len(bbox) >= 4:
        return [float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])]
    return None


def _stable_value(raw_value, previous_value, smoothing=0.9, max_step_fraction=0.08):
    raw_value = float(raw_value)
    if previous_value is None or previous_value <= 0:
        return raw_value
    lo = previous_value * (1.0 - float(max_step_fraction))
    hi = previous_value * (1.0 + float(max_step_fraction))
    clipped = float(np.clip(raw_value, lo, hi))
    return previous_value * float(smoothing) + clipped * (1.0 - float(smoothing))


def _stable_position(raw_value, previous_value, reference_size, smoothing=0.78, max_step_fraction=0.12):
    raw_value = float(raw_value)
    if previous_value is None:
        return raw_value
    max_step = max(1.0, float(reference_size) * float(max_step_fraction))
    clipped = float(np.clip(raw_value, previous_value - max_step, previous_value + max_step))
    return previous_value * float(smoothing) + clipped * (1.0 - float(smoothing))


def _downsample_gray_for_character_floor(frame_bgr, frame_width):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    analysis_width = int(CHARACTER_FLOOR_MOTION_ANALYSIS_WIDTH)
    if analysis_width > 0 and frame_width > analysis_width:
        scale = analysis_width / float(frame_width)
        gray = cv2.resize(gray, (analysis_width, max(1, int(round(gray.shape[0] * scale)))), interpolation=cv2.INTER_AREA)
        return gray, 1.0 / scale
    return gray, 1.0


def _estimate_floor_motion(prev_gray, curr_gray, person, frame_width, frame_height, motion_scale=1.0):
    """画面下部の背景特徴点から、床/背景が画像内でどれだけ動いたかを推定します。"""
    if not CHARACTER_FLOOR_MOTION_ENABLED or prev_gray is None or curr_gray is None:
        return 0.0, 0.0
    gh, gw = curr_gray.shape[:2]
    roi_top = int(gh * (1.0 - float(CHARACTER_FLOOR_ROI_BOTTOM_FRACTION)))
    mask = np.zeros_like(prev_gray, dtype=np.uint8)
    mask[roi_top:gh, :] = 255
    bbox = person.get('bbox_xyxy') if person else None
    if bbox and len(bbox) >= 4:
        inv_scale = 1.0 / max(1e-6, float(motion_scale))
        x1, y1, x2, y2 = [int(float(v) * inv_scale) for v in bbox[:4]]
        pad = int(max(8, gh * 0.035))
        cv2.rectangle(mask, (max(0, x1 - pad), max(roi_top, y1 - pad)), (min(gw - 1, x2 + pad), min(gh - 1, y2 + pad)), 0, -1)
    pts0 = cv2.goodFeaturesToTrack(prev_gray, maxCorners=int(CHARACTER_FLOOR_MOTION_MAX_CORNERS), qualityLevel=0.015, minDistance=14, blockSize=7, mask=mask)
    if pts0 is None or len(pts0) < 6:
        return 0.0, 0.0
    pts1, status, _err = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, pts0, None, winSize=(17, 17), maxLevel=2)
    if pts1 is None or status is None:
        return 0.0, 0.0
    good0 = pts0[status.reshape(-1) == 1].reshape(-1, 2)
    good1 = pts1[status.reshape(-1) == 1].reshape(-1, 2)
    if len(good0) < 6:
        return 0.0, 0.0
    delta = (good1 - good0) * float(motion_scale)
    dist = np.linalg.norm(delta, axis=1)
    limit = max(2.0, frame_height * float(CHARACTER_FLOOR_MOTION_MAX_STEP_FRACTION))
    delta = delta[dist <= limit]
    if len(delta) < 4:
        return 0.0, 0.0
    dx, dy = np.median(delta, axis=0)
    return float(dx) * float(CHARACTER_FLOOR_MOTION_GAIN), float(dy) * float(CHARACTER_FLOOR_MOTION_GAIN)


def _apply_floor_motion_to_anchor(previous_anchor_state, floor_dx, floor_dy):
    if not previous_anchor_state:
        return previous_anchor_state
    return {
        'x': float(previous_anchor_state.get('x', 0.0)) + float(floor_dx),
        'bottom_y': float(previous_anchor_state.get('bottom_y', 0.0)) + float(floor_dy),
    }


def _stabilize_character_anchor(raw_x, raw_bottom_y, previous_anchor_state, reference_height):
    """人形全体の足元アンカーを安定させ、床を滑るような移動を抑えます。"""
    if not CHARACTER_FOOT_LOCK_ENABLED:
        return int(round(raw_x)), int(round(raw_bottom_y)), {'x': float(raw_x), 'bottom_y': float(raw_bottom_y)}
    prev_x = previous_anchor_state.get('x') if previous_anchor_state else None
    prev_y = previous_anchor_state.get('bottom_y') if previous_anchor_state else None
    x = _stable_position(
        raw_x,
        prev_x,
        reference_height,
        smoothing=CHARACTER_ANCHOR_X_SMOOTHING,
        max_step_fraction=CHARACTER_ANCHOR_X_MAX_STEP,
    )
    raw_bottom_y = float(raw_bottom_y)
    if prev_y is None:
        y = raw_bottom_y
    else:
        # 画像座標ではyが小さいほど上、大きいほど下。上方向は強く制限し、接地の浮き上がりを防ぎます。
        up_step = max(1.0, float(reference_height) * float(CHARACTER_ANCHOR_Y_UP_MAX_STEP))
        down_step = max(1.0, float(reference_height) * float(CHARACTER_ANCHOR_Y_DOWN_MAX_STEP))
        clipped_y = float(np.clip(raw_bottom_y, prev_y - up_step, prev_y + down_step))
        y = prev_y * float(CHARACTER_ANCHOR_Y_SMOOTHING) + clipped_y * (1.0 - float(CHARACTER_ANCHOR_Y_SMOOTHING))
    return int(round(x)), int(round(y)), {'x': float(x), 'bottom_y': float(y)}


def _downsample_gray_for_world(frame_bgr, frame_width):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    analysis_width = int(CHARACTER_WORLD_MOTION_ANALYSIS_WIDTH)
    if analysis_width > 0 and frame_width > analysis_width:
        scale = analysis_width / float(frame_width)
        gray = cv2.resize(gray, (analysis_width, max(1, int(round(gray.shape[0] * scale)))), interpolation=cv2.INTER_AREA)
        return gray, 1.0 / scale
    return gray, 1.0


def _estimate_world_homography(prev_gray, curr_gray, person, motion_scale=1.0):
    """床・壁など背景全体の特徴点から、カメラ運動をホモグラフィとして推定します(フル解像度座標系)。

    人物bbox内の特徴点は除外し、静止した背景(床・壁・家具など)だけを使います。
    返り値は前フレーム座標→現フレーム座標の3x3行列。推定失敗時は None。
    """
    if prev_gray is None or curr_gray is None or prev_gray.shape != curr_gray.shape:
        return None
    gh, gw = prev_gray.shape[:2]
    mask = np.full((gh, gw), 255, dtype=np.uint8)
    bbox = person.get('bbox_xyxy') if person else None
    if bbox and len(bbox) >= 4:
        inv_scale = 1.0 / max(1e-6, float(motion_scale))
        x1, y1, x2, y2 = [int(float(v) * inv_scale) for v in bbox[:4]]
        pad = int(max(8, gh * 0.04))
        cv2.rectangle(mask, (max(0, x1 - pad), max(0, y1 - pad)), (min(gw - 1, x2 + pad), min(gh - 1, y2 + pad)), 0, -1)
    pts0 = cv2.goodFeaturesToTrack(
        prev_gray,
        maxCorners=int(CHARACTER_WORLD_MOTION_MAX_CORNERS),
        qualityLevel=0.01,
        minDistance=10,
        blockSize=7,
        mask=mask,
    )
    if pts0 is None or len(pts0) < int(CHARACTER_WORLD_MOTION_MIN_INLIERS):
        return None
    pts1, status, _err = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, pts0, None, winSize=(21, 21), maxLevel=3)
    if pts1 is None or status is None:
        return None
    good0 = pts0[status.reshape(-1) == 1].reshape(-1, 2) * float(motion_scale)
    good1 = pts1[status.reshape(-1) == 1].reshape(-1, 2) * float(motion_scale)
    if len(good0) < int(CHARACTER_WORLD_MOTION_MIN_INLIERS):
        return None
    H, inlier_mask = cv2.findHomography(good0, good1, cv2.RANSAC, float(CHARACTER_WORLD_MOTION_RANSAC_REPROJ))
    if H is None or inlier_mask is None or int(inlier_mask.sum()) < int(CHARACTER_WORLD_MOTION_MIN_INLIERS):
        return None
    if not np.all(np.isfinite(H)) or abs(H[2, 2]) < 1e-8:
        return None
    return H


def _apply_homography_point(H, x, y):
    p = H @ np.array([float(x), float(y), 1.0], dtype=np.float64)
    if abs(p[2]) < 1e-9:
        return float(x), float(y)
    return float(p[0] / p[2]), float(p[1] / p[2])


def _homography_local_scale(H, x, y, d=8.0):
    # アンカー付近の微小変位がどれだけ拡大/縮小されるかを数値的に測り、カメラズーム率とします。
    x0, y0 = _apply_homography_point(H, x, y)
    x1, y1 = _apply_homography_point(H, x + d, y)
    x2, y2 = _apply_homography_point(H, x, y + d)
    area = abs((x1 - x0) * (y2 - y0) - (x2 - x0) * (y1 - y0))
    return math.sqrt(max(1e-9, area)) / d


def _update_world_locked_anchor(state, H, person, frame_w, frame_h, base_height):
    """世界固定アンカーをカメラ運動で追従させ、必要なら人物相対の目標へごくゆっくり引き戻します。"""
    if state is None:
        tx, ty = _character_anchor_for_person(frame_w, frame_h, person, base_height)
        state = {'x': float(tx), 'bottom_y': float(ty), 'height': float(base_height)}
        return int(round(state['x'])), int(round(state['bottom_y'])), float(state['height']), state
    x, y, h_px = state['x'], state['bottom_y'], state['height']
    if H is not None:
        if CHARACTER_WORLD_SCALE_WITH_CAMERA:
            s = _homography_local_scale(H, x, y)
            step = float(CHARACTER_WORLD_SCALE_MAX_STEP)
            h_px *= float(np.clip(s, 1.0 - step, 1.0 + step))
        x, y = _apply_homography_point(H, x, y)
    gain = float(CHARACTER_WORLD_LOCK_REATTACH_GAIN)
    if gain > 0.0 and person is not None:
        tx, ty = _character_anchor_for_person(frame_w, frame_h, person, h_px)
        x += (float(tx) - x) * gain
        y += (float(ty) - y) * gain
    # 暴走防止: 画面から大きく外れすぎない範囲に収め、サイズも常識的な範囲に保ちます。
    offscreen = h_px * float(CHARACTER_ALLOW_PARTIAL_OFFSCREEN_FRACTION)
    x = float(np.clip(x, -offscreen, frame_w + offscreen))
    y = float(np.clip(y, h_px - offscreen, frame_h + offscreen * 0.55))
    h_px = float(np.clip(h_px, base_height * 0.4, base_height * 2.5))
    state = {'x': x, 'bottom_y': y, 'height': h_px}
    return int(round(x)), int(round(y)), float(h_px), state



def _build_pose_mapper(kp_dict, person, target_x, target_bottom_y, target_height, previous_mapper_state=None):
    bounds = _pose_bounds(kp_dict, person)
    if not bounds:
        return None
    x1, y1, x2, y2 = bounds
    src_w = max(1.0, x2 - x1)
    raw_src_h = max(1.0, y2 - y1)
    prev_src_h = previous_mapper_state.get('src_h') if previous_mapper_state else None
    src_h = _stable_value(
        raw_src_h,
        prev_src_h,
        smoothing=CHARACTER_SOURCE_HEIGHT_SMOOTHING,
        max_step_fraction=CHARACTER_SOURCE_HEIGHT_MAX_STEP,
    )
    raw_scale = float(target_height) / src_h
    prev_scale = previous_mapper_state.get('scale') if previous_mapper_state else None
    scale = _stable_value(
        raw_scale,
        prev_scale,
        smoothing=CHARACTER_SCALE_SMOOTHING,
        max_step_fraction=CHARACTER_SCALE_MAX_STEP,
    )
    raw_src_cx = (x1 + x2) * 0.5
    raw_src_bottom = y2
    prev_src_cx = previous_mapper_state.get('src_cx') if previous_mapper_state else None
    prev_src_bottom = previous_mapper_state.get('src_bottom') if previous_mapper_state else None
    src_cx = _stable_position(
        raw_src_cx,
        prev_src_cx,
        src_h,
        smoothing=CHARACTER_SOURCE_POSITION_SMOOTHING,
        max_step_fraction=CHARACTER_SOURCE_POSITION_MAX_STEP,
    )
    src_bottom = _stable_position(
        raw_src_bottom,
        prev_src_bottom,
        src_h,
        smoothing=CHARACTER_SOURCE_POSITION_SMOOTHING,
        max_step_fraction=CHARACTER_SOURCE_POSITION_MAX_STEP,
    )
    mapper_state = {
        'src_h': float(src_h),
        'raw_src_h': float(raw_src_h),
        'scale': float(scale),
        'raw_scale': float(raw_scale),
        'src_cx': float(src_cx),
        'src_bottom': float(src_bottom),
        'raw_src_cx': float(raw_src_cx),
        'raw_src_bottom': float(raw_src_bottom),
    }

    def map_xy(pt):
        return np.array(
            [
                target_x + (float(pt[0]) - src_cx) * scale,
                target_bottom_y - (src_bottom - float(pt[1])) * scale,
            ],
            dtype=np.float32,
        )

    return map_xy, scale, mapper_state


def _smooth_pose(current_pose, previous_pose, alpha):
    if previous_pose is None or alpha <= 0:
        return current_pose
    smoothed = {}
    for name, point in current_pose.items():
        if name in previous_pose:
            smoothed[name] = previous_pose[name] * float(alpha) + point * (1.0 - float(alpha))
        else:
            smoothed[name] = point
    return smoothed


def _draw_capsule(layer, p1, p2, color_bgra, thickness):
    if p1 is None or p2 is None:
        return
    p1i = tuple(np.round(p1).astype(int))
    p2i = tuple(np.round(p2).astype(int))
    cv2.line(layer, p1i, p2i, color_bgra, int(thickness), cv2.LINE_AA)
    cv2.circle(layer, p1i, max(1, int(thickness / 2)), color_bgra, -1, cv2.LINE_AA)
    cv2.circle(layer, p2i, max(1, int(thickness / 2)), color_bgra, -1, cv2.LINE_AA)


def _draw_joint(layer, p, radius, color_bgra):
    if p is None:
        return
    cv2.circle(layer, tuple(np.round(p).astype(int)), int(radius), color_bgra, -1, cv2.LINE_AA)


def _draw_polygon(layer, points, color_bgra):
    valid = [p for p in points if p is not None]
    if len(valid) < 3:
        return
    pts = np.array([np.round(p).astype(int) for p in valid], dtype=np.int32)
    cv2.fillConvexPoly(layer, pts, color_bgra, cv2.LINE_AA)


def _candidate_head_image_paths():
    paths = []
    if CHARACTER_HEAD_IMAGE_PATH:
        paths.append(Path(CHARACTER_HEAD_IMAGE_PATH))
    paths.extend([
        Path('/content') / CHARACTER_HEAD_IMAGE_FILENAME,
        CHARACTER_HEAD_IMAGE_DRIVE_DIR / CHARACTER_HEAD_IMAGE_FILENAME,
        Path.cwd() / 'assets' / CHARACTER_HEAD_IMAGE_FILENAME,
        Path.cwd() / 'rf-detr' / 'docs' / 'cookbooks' / 'assets' / CHARACTER_HEAD_IMAGE_FILENAME,
        Path(r'C:/work/code/ER-FlowScan/rf-detr/docs/cookbooks/assets') / CHARACTER_HEAD_IMAGE_FILENAME,
        Path(r'C:/work/code/Dancing-character') / CHARACTER_HEAD_IMAGE_FILENAME,
        Path(r'C:\work\code\Dancing-character') / CHARACTER_HEAD_IMAGE_FILENAME,
    ])
    seen = set()
    unique = []
    for path in paths:
        key = str(path)
        if key not in seen:
            seen.add(key)
            unique.append(path)
    return unique


def _ensure_head_image_path():
    for path in _candidate_head_image_paths():
        if path.exists():
            return path
    try:
        from google.colab import files
        print(f'頭画像をアップロードしてください: {CHARACTER_HEAD_IMAGE_FILENAME}')
        uploaded = files.upload()
        if uploaded:
            source_name = CHARACTER_HEAD_IMAGE_FILENAME if CHARACTER_HEAD_IMAGE_FILENAME in uploaded else next(iter(uploaded.keys()))
            target = Path('/content') / CHARACTER_HEAD_IMAGE_FILENAME
            target.write_bytes(uploaded[source_name])
            try:
                from google.colab import drive
                if not Path('/content/drive/MyDrive').is_dir():
                    drive.mount('/content/drive', force_remount=False)
                CHARACTER_HEAD_IMAGE_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
                drive_target = CHARACTER_HEAD_IMAGE_DRIVE_DIR / CHARACTER_HEAD_IMAGE_FILENAME
                drive_target.write_bytes(target.read_bytes())
                print(f'頭画像を次回用にDriveへ保存しました: {drive_target}')
            except Exception as exc:
                print(f'Driveへの頭画像保存はスキップしました: {exc}')
            return target
    except Exception as exc:
        print(f'頭画像アップロードは使えませんでした: {exc}')
    return None


def _prepare_head_image_bgra():
    path = _ensure_head_image_path()
    if path is None:
        print('頭画像が見つからないため、従来の丸顔を描画します。')
        return None
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        print(f'頭画像を読めませんでした。従来の丸顔を描画します: {path}')
        return None
    if CHARACTER_HEAD_IMAGE_ALPHA_MODE == 'square':
        alpha = np.full(image_bgr.shape[:2], 255, dtype=np.uint8)
    else:
        max_ch = image_bgr.max(axis=2).astype(np.float32)
        alpha = ((max_ch - float(CHARACTER_HEAD_IMAGE_DARK_THRESHOLD)) * (255.0 / max(1.0, 255.0 - float(CHARACTER_HEAD_IMAGE_DARK_THRESHOLD))))
        alpha = np.clip(alpha, 0, 255).astype(np.uint8)
        kernel = np.ones((3, 3), np.uint8)
        alpha = cv2.morphologyEx(alpha, cv2.MORPH_OPEN, kernel)
        alpha = cv2.GaussianBlur(alpha, (0, 0), 1.2)
        ys, xs = np.where(alpha > 8)
        if len(xs) > 0 and len(ys) > 0:
            pad = 10
            x1 = max(0, int(xs.min()) - pad)
            x2 = min(image_bgr.shape[1], int(xs.max()) + pad + 1)
            y1 = max(0, int(ys.min()) - pad)
            y2 = min(image_bgr.shape[0], int(ys.max()) + pad + 1)
            image_bgr = image_bgr[y1:y2, x1:x2]
            alpha = alpha[y1:y2, x1:x2]
    bgra = np.dstack([image_bgr, alpha]).astype(np.uint8)
    print(f'head image loaded: {path} size={bgra.shape[1]}x{bgra.shape[0]} mode={CHARACTER_HEAD_IMAGE_ALPHA_MODE}')
    return bgra


CHARACTER_HEAD_IMAGE_BGRA = _prepare_head_image_bgra()


def _overlay_bgra_at_anchor(layer, sprite_bgra, anchor_x, anchor_y, anchor_u=0.5, anchor_v=0.5, opacity=1.0):
    if sprite_bgra is None or sprite_bgra.size == 0:
        return
    h, w = sprite_bgra.shape[:2]
    x1 = int(round(anchor_x - w * anchor_u))
    y1 = int(round(anchor_y - h * anchor_v))
    x2 = x1 + w
    y2 = y1 + h
    lx1, ly1 = max(0, x1), max(0, y1)
    lx2, ly2 = min(layer.shape[1], x2), min(layer.shape[0], y2)
    if lx1 >= lx2 or ly1 >= ly2:
        return
    sx1, sy1 = lx1 - x1, ly1 - y1
    sx2, sy2 = sx1 + (lx2 - lx1), sy1 + (ly2 - ly1)
    sprite = sprite_bgra[sy1:sy2, sx1:sx2].astype(np.float32)
    dst = layer[ly1:ly2, lx1:lx2].astype(np.float32)
    alpha = (sprite[:, :, 3:4] / 255.0) * float(opacity)
    out_rgb = sprite[:, :, :3] * alpha + dst[:, :, :3] * (1.0 - alpha)
    out_alpha = np.clip(sprite[:, :, 3:4] * float(opacity) + dst[:, :, 3:4] * (1.0 - alpha), 0, 255)
    dst[:, :, :3] = out_rgb
    dst[:, :, 3:4] = out_alpha
    layer[ly1:ly2, lx1:lx2] = np.clip(dst, 0, 255).astype(np.uint8)


CHARACTER_HEAD_ANGLE_STATE = {'angle': None}


def _unwrap_angle_near(raw_angle, previous_angle):
    raw_angle = float(raw_angle)
    if previous_angle is None:
        return raw_angle
    diff = (raw_angle - previous_angle + 180.0) % 360.0 - 180.0
    return previous_angle + diff


def _smooth_head_angle(raw_angle):
    limit = abs(float(CHARACTER_HEAD_IMAGE_MAX_ROTATION_DEG))
    raw_angle = float(np.clip(raw_angle, -limit, limit))
    previous = CHARACTER_HEAD_ANGLE_STATE.get('angle')
    if previous is None:
        CHARACTER_HEAD_ANGLE_STATE['angle'] = raw_angle
        return raw_angle
    raw_angle = _unwrap_angle_near(raw_angle, previous)
    step = abs(float(CHARACTER_HEAD_IMAGE_MAX_ROTATION_STEP_DEG))
    raw_angle = float(np.clip(raw_angle, previous - step, previous + step))
    smoothed = previous * float(CHARACTER_HEAD_IMAGE_ROTATION_SMOOTHING) + raw_angle * (1.0 - float(CHARACTER_HEAD_IMAGE_ROTATION_SMOOTHING))
    smoothed = float(np.clip(smoothed, -limit, limit))
    CHARACTER_HEAD_ANGLE_STATE['angle'] = smoothed
    return smoothed


def _angle_between_points(a, b):
    if a is None or b is None:
        return None
    return math.degrees(math.atan2(float(b[1] - a[1]), float(b[0] - a[0])))


def _estimate_head_image_angle(nose, leye, reye, lear, rear, ls, rs, target_height):
    mode = CHARACTER_HEAD_IMAGE_ROTATION_MODE
    if CHARACTER_HEAD_IMAGE_ROTATE_WITH_SHOULDERS and mode == 'off':
        mode = 'shoulders'
    if mode == 'off':
        CHARACTER_HEAD_ANGLE_STATE['angle'] = _smooth_head_angle(0.0)
        return CHARACTER_HEAD_ANGLE_STATE['angle']

    angle_candidates = []
    # 顔のロール角: 両目が見えていれば目線、両耳が見えていれば耳線を使う。
    if mode == 'face_roll':
        eye_angle = _angle_between_points(leye, reye)
        if eye_angle is not None:
            angle_candidates.append((eye_angle, 1.0))
        ear_angle = _angle_between_points(lear, rear)
        if ear_angle is not None:
            angle_candidates.append((ear_angle, 0.65))

    # 顔点が弱い/ない時の保険: 肩傾きを弱く反映。首支点の自然な傾き用。
    shoulder_angle = _angle_between_points(ls, rs)
    if shoulder_angle is not None:
        shoulder_weight = 0.35 if mode == 'face_roll' else 1.0
        angle_candidates.append((shoulder_angle, shoulder_weight))

    if not angle_candidates:
        return _smooth_head_angle(0.0)

    weighted = sum(angle * weight for angle, weight in angle_candidates) / max(1e-6, sum(weight for _, weight in angle_candidates))

    # 振り返り/横向きの簡易検知: 鼻が目の中心から左右へ寄るほど、軽い回転補正を足す。
    # 2Dだけではyawを正確に復元できないので、控えめな演出に留める。
    if nose is not None and leye is not None and reye is not None:
        eye_mid = (leye + reye) * 0.5
        eye_span = max(1.0, float(np.linalg.norm(reye - leye)))
        turn_ratio = float(np.clip((nose[0] - eye_mid[0]) / eye_span, -1.0, 1.0))
        weighted += turn_ratio * float(CHARACTER_HEAD_TURN_ROTATION_DEG)

    return _smooth_head_angle(weighted)


def _rotated_resized_head_sprite(head_bgra, target_size, angle_deg=0.0):
    if head_bgra is None:
        return None
    h, w = head_bgra.shape[:2]
    scale = float(target_size) / max(1, max(h, w))
    new_w = max(2, int(round(w * scale)))
    new_h = max(2, int(round(h * scale)))
    sprite = cv2.resize(head_bgra, (new_w, new_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
    if abs(angle_deg) < 0.1:
        return sprite

    # 首に相当する画像下部アンカーを回転支点にする。
    # CHARACTER_HEAD_IMAGE_ANCHOR_Y が 1.0 以上なら、画像下端付近/下端外側を支点にできます。
    pivot = (new_w / 2.0, new_h * float(CHARACTER_HEAD_IMAGE_ANCHOR_Y))
    mat = cv2.getRotationMatrix2D(pivot, float(angle_deg), 1.0)
    corners = np.array([[0, 0, 1], [new_w, 0, 1], [new_w, new_h, 1], [0, new_h, 1]], dtype=np.float32)
    transformed = corners @ mat.T
    min_xy = transformed.min(axis=0)
    max_xy = transformed.max(axis=0)
    bound_w = max(2, int(math.ceil(max_xy[0] - min_xy[0])))
    bound_h = max(2, int(math.ceil(max_xy[1] - min_xy[1])))
    mat[0, 2] -= min_xy[0]
    mat[1, 2] -= min_xy[1]
    rotated = cv2.warpAffine(sprite, mat, (bound_w, bound_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
    return rotated


def _draw_default_head(layer, head_center, head_r, nose, ls, rs, skin, hair, outline):
    c = tuple(np.round(head_center).astype(int))
    cv2.circle(layer, c, int(head_r * 1.10), hair, -1, cv2.LINE_AA)
    cv2.circle(layer, (c[0], c[1] + int(head_r * 0.08)), head_r, skin, -1, cv2.LINE_AA)
    if nose is not None:
        eye_y = int(c[1] - head_r * 0.12)
        eye_dx = max(2, int(head_r * 0.33))
        cv2.circle(layer, (c[0] - eye_dx, eye_y), max(1, int(head_r * 0.08)), outline, -1, cv2.LINE_AA)
        cv2.circle(layer, (c[0] + eye_dx, eye_y), max(1, int(head_r * 0.08)), outline, -1, cv2.LINE_AA)
        cv2.ellipse(layer, (c[0], c[1] + int(head_r * 0.22)), (max(2, int(head_r * 0.28)), max(1, int(head_r * 0.10))), 0, 0, 180, outline, max(1, int(head_r * 0.05)), cv2.LINE_AA)
def _draw_colored_character(frame_shape, mapped, target_height, z_map=None):
    """キーポイント人形を1枚のBGRAレイヤーに描画します。

    z_map(MediaPipe 3Dのz値。小さいほどカメラに近い)があるフレームでは、胴体・四肢・頭を
    奥行き順に並べ替えて描画し、手が顔の前/後ろを実際の深度で切り替えます。
    z_mapが無いフレームは従来の固定順(+CHARACTER_HANDS_OVER_FACE)で描画します。
    胴体は肩と腰の横ズレから「腰のくねり」を曲がった背骨と6点ポリゴンで表現します。
    """
    h, w = frame_shape[:2]
    layer = np.zeros((h, w, 4), dtype=np.uint8)

    # COCO / YOLO-Pose / MediaPipe common names.
    nose = mapped.get('nose')
    leye = mapped.get('left_eye')
    reye = mapped.get('right_eye')
    lear = mapped.get('left_ear')
    rear = mapped.get('right_ear')
    ls = mapped.get('left_shoulder')
    rs = mapped.get('right_shoulder')
    le = mapped.get('left_elbow')
    re = mapped.get('right_elbow')
    lw = mapped.get('left_wrist')
    rw = mapped.get('right_wrist')
    lh = mapped.get('left_hip')
    rh = mapped.get('right_hip')
    lk = mapped.get('left_knee')
    rk = mapped.get('right_knee')
    la = mapped.get('left_ankle')
    ra = mapped.get('right_ankle')

    # 腰の横揺れ(ヒップスウェイ)の誇張: 肩中心に対する腰中心の横ズレを増幅します。
    # 膝は60%連動させ、足首はそのままにすることで、足を床に着けたまま腰から上が
    # くねる自然なS字カーブになります。
    _sway_gain = float(CHARACTER_HIP_SWAY_GAIN)
    if _sway_gain != 1.0 and ls is not None and rs is not None and lh is not None and rh is not None:
        _neck_x = float((ls[0] + rs[0]) * 0.5)
        _hip_x = float((lh[0] + rh[0]) * 0.5)
        _extra = np.array([(_hip_x - _neck_x) * (_sway_gain - 1.0), 0.0], dtype=np.float32)
        lh = lh + _extra
        rh = rh + _extra
        if lk is not None:
            lk = lk + _extra * 0.6
        if rk is not None:
            rk = rk + _extra * 0.6


    line = max(4, int(target_height * 0.045 * CHARACTER_LINE_SCALE))
    arm_line = max(3, int(line * 0.82))
    leg_line = max(4, int(line * 0.95))
    joint_r = max(3, int(line * 0.46))

    skin = (104, 178, 255, 245)      # BGRA
    shirt = (76, 132, 238, 245)
    shirt_dark = (42, 83, 174, 245)
    pants = (198, 94, 74, 245)
    pants_dark = (132, 59, 50, 245)
    shoe = (36, 38, 46, 245)
    hair = (42, 35, 30, 245)
    white = (250, 250, 250, 245)
    outline = (22, 24, 30, 210)

    have_z = bool(z_map) and bool(CHARACTER_DEPTH_SORT_ENABLED)

    def seg_z(names, fallback):
        # zあり: 実測深度(負=カメラ側)。zなし: 従来の描画順を再現する擬似深度。
        return _z_of(z_map, *names, default=fallback) if have_z else float(fallback)

    # zなしフレームの擬似深度(大きい=奥)。従来: 胴体→右(奥側)→左(手前側)→頭→(設定により)手を最前面。
    z_torso_fb = 3.0
    z_rleg_fb, z_lleg_fb = 2.2, 2.0
    if CHARACTER_HANDS_OVER_FACE:
        z_rarm_fb, z_larm_fb = -2.0, -2.1  # 頭(-1.0)より手前
    else:
        z_rarm_fb, z_larm_fb = 2.4, 1.8    # 頭より奥
    z_head_fb = -1.0

    draw_ops = []

    # --- 胴体: 腰のくねり付き ---
    def _draw_torso():
        if ls is None or rs is None or lh is None or rh is None:
            _draw_polygon(layer, [ls, rs, rh, lh], shirt)
            if ls is not None and rs is not None and lh is not None and rh is not None:
                _draw_capsule(layer, (ls + rs) * 0.5, (lh + rh) * 0.5, shirt_dark, max(3, int(line * 0.55)))
            return
        neck = (ls + rs) * 0.5
        hipc = (lh + rh) * 0.5
        gain = float(CHARACTER_TORSO_BEND_GAIN)
        # 肩中心と腰中心の横ズレ(=体のくねり)をウエスト中間点へ反映し、gainで誇張します。
        sway_x = float(hipc[0] - neck[0])
        bend = np.array([sway_x * (gain - 1.0), 0.0], dtype=np.float32)
        mid_l = (ls + lh) * 0.5 + bend
        mid_r = (rs + rh) * 0.5 + bend
        pts = np.array([np.round(p).astype(int) for p in [ls, rs, mid_r, rh, lh, mid_l]], dtype=np.int32)
        cv2.fillPoly(layer, [pts], shirt, cv2.LINE_AA)
        # 肩・腰のバーで体に丸みを付けます。
        _draw_capsule(layer, ls, rs, shirt, max(3, int(line * 0.60)))
        _draw_capsule(layer, lh, rh, pants, max(3, int(line * 0.70)))
        # 曲がった背骨: 首→ウエスト(くねり反映)→腰 の2本カプセル。
        mid_bent = (neck + hipc) * 0.5 + bend
        _draw_capsule(layer, neck, mid_bent, shirt_dark, max(3, int(line * 0.55)))
        _draw_capsule(layer, mid_bent, hipc, shirt_dark, max(3, int(line * 0.55)))

    draw_ops.append((seg_z(['left_shoulder', 'right_shoulder', 'left_hip', 'right_hip'], z_torso_fb), _draw_torso))

    def _mk_capsule(a, b, color, thick):
        return lambda: _draw_capsule(layer, a, b, color, thick)

    def _mk_joint(p, r, color):
        return lambda: _draw_joint(layer, p, r, color)

    def _mk_joint_ring(p):
        def _fn():
            _draw_joint(layer, p, joint_r, outline)
            _draw_joint(layer, p, max(1, joint_r - 2), white)
        return _fn

    # --- 四肢: セグメント単位で深度を持たせます ---
    _segments = [
        (['right_shoulder', 'right_elbow'], rs, re, skin, arm_line, z_rarm_fb),
        (['right_elbow', 'right_wrist'], re, rw, skin, arm_line, z_rarm_fb),
        (['left_shoulder', 'left_elbow'], ls, le, skin, arm_line, z_larm_fb),
        (['left_elbow', 'left_wrist'], le, lw, skin, arm_line, z_larm_fb),
        (['right_hip', 'right_knee'], rh, rk, pants_dark, leg_line, z_rleg_fb),
        (['right_knee', 'right_ankle'], rk, ra, pants_dark, leg_line, z_rleg_fb),
        (['left_hip', 'left_knee'], lh, lk, pants, leg_line, z_lleg_fb),
        (['left_knee', 'left_ankle'], lk, la, pants, leg_line, z_lleg_fb),
    ]
    for names, a, b, color, thick, fb in _segments:
        if a is None or b is None:
            continue
        draw_ops.append((seg_z(names, fb), _mk_capsule(a, b, color, thick)))

    # 手先・足先(対応セグメントよりわずかに手前)。
    for names, p, r, color, fb in [
        (['left_wrist'], lw, joint_r + 1, skin, z_larm_fb),
        (['right_wrist'], rw, joint_r + 1, skin, z_rarm_fb),
        (['left_ankle'], la, joint_r + 2, shoe, z_lleg_fb),
        (['right_ankle'], ra, joint_r + 2, shoe, z_rleg_fb),
    ]:
        if p is None:
            continue
        draw_ops.append((seg_z(names, fb) - 0.01, _mk_joint(p, r, color)))

    # 関節リング。
    for p, names, fb in [
        (ls, ['left_shoulder', 'left_elbow'], z_larm_fb), (rs, ['right_shoulder', 'right_elbow'], z_rarm_fb),
        (le, ['left_elbow', 'left_wrist'], z_larm_fb), (re, ['right_elbow', 'right_wrist'], z_rarm_fb),
        (lh, ['left_hip', 'left_knee'], z_lleg_fb), (rh, ['right_hip', 'right_knee'], z_rleg_fb),
        (lk, ['left_knee', 'left_ankle'], z_lleg_fb), (rk, ['right_knee', 'right_ankle'], z_rleg_fb),
    ]:
        if p is None:
            continue
        draw_ops.append((seg_z(names, fb) - 0.03, _mk_joint_ring(p)))

    # --- 頭(画像 or 丸顔): 深度は顔ランドマークの平均 ---
    def _draw_head():
        base_head_r = target_height * float(CHARACTER_HEAD_RADIUS_FRACTION) * float(CHARACTER_HEAD_SCALE)
        min_head_r = target_height * float(CHARACTER_HEAD_RADIUS_MIN_FRACTION) * float(CHARACTER_HEAD_SCALE)
        max_head_r = target_height * float(CHARACTER_HEAD_RADIUS_MAX_FRACTION) * float(CHARACTER_HEAD_SCALE)
        head_r = int(np.clip(base_head_r, min_head_r, max_head_r))
        face_points = [p for p in [nose, leye, reye, lear, rear] if p is not None]
        if face_points:
            face = np.array(face_points, dtype=np.float32)
            head_center = face.mean(axis=0)
            if ls is not None and rs is not None:
                shoulder_mid = (ls + rs) * 0.5
                expected_head = shoulder_mid + np.array([0, -target_height * 0.12], dtype=np.float32)
                max_face_offset = target_height * 0.12
                delta = head_center - expected_head
                dist = float(np.linalg.norm(delta))
                if dist > max_face_offset:
                    head_center = expected_head + delta * (max_face_offset / max(dist, 1e-6))
                head_center = head_center * 0.45 + expected_head * 0.55
        elif ls is not None and rs is not None:
            shoulder_mid = (ls + rs) * 0.5
            head_center = shoulder_mid + np.array([0, -target_height * 0.13], dtype=np.float32)
        else:
            return
        if head_r <= 0:
            return
        head_angle = _estimate_head_image_angle(nose, leye, reye, lear, rear, ls, rs, target_height)
        if CHARACTER_HEAD_IMAGE_BGRA is not None:
            target_size = max(8, int(head_r * 2.0 * float(CHARACTER_HEAD_IMAGE_SCALE)))
            head_sprite = _rotated_resized_head_sprite(CHARACTER_HEAD_IMAGE_BGRA, target_size, head_angle)
            if CHARACTER_HEAD_IMAGE_ATTACH_MODE == 'neck' and ls is not None and rs is not None:
                neck_anchor = (ls + rs) * 0.5 + np.array([0.0, target_height * float(CHARACTER_HEAD_IMAGE_NECK_OFFSET_FRACTION)], dtype=np.float32)
                anchor_x, anchor_y = float(neck_anchor[0]), float(neck_anchor[1])
            else:
                anchor_x, anchor_y = float(head_center[0]), float(head_center[1])
            _overlay_bgra_at_anchor(
                layer,
                head_sprite,
                anchor_x,
                anchor_y,
                anchor_u=0.5,
                anchor_v=float(CHARACTER_HEAD_IMAGE_ANCHOR_Y),
                opacity=CHARACTER_HEAD_IMAGE_OPACITY,
            )
        else:
            _draw_default_head(layer, head_center, head_r, nose, ls, rs, skin, hair, outline)

    draw_ops.append((seg_z(['nose', 'left_eye', 'right_eye'], z_head_fb), _draw_head))

    # --- 奥から手前の順に描画(zが大きい=奥を先に) ---
    for _z, _fn in sorted(draw_ops, key=lambda item: -item[0]):
        _fn()

    return layer


def _map_pose_for_character(kp_dict, person, target_x, target_bottom_y, target_height, previous_pose=None, previous_mapper_state=None):
    mapper = _build_pose_mapper(kp_dict, person, target_x, target_bottom_y, target_height, previous_mapper_state)
    if mapper is None:
        return None, previous_mapper_state
    map_xy, _scale, mapper_state = mapper
    mapped = {}
    for name, value in kp_dict.items():
        mapped[name] = map_xy([value[0], value[1]])
    return _smooth_pose(mapped, previous_pose, CHARACTER_SMOOTHING), mapper_state


rigged_raw_path = OUTPUT_DIR / 'in_frame_character_rigged_raw.mp4'
rigged_final_path = OUTPUT_DIR / 'in_frame_character_rigged.mp4'

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f'Could not open video: {video_path}')

writer = cv2.VideoWriter(
    str(rigged_raw_path),
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps / max(FRAME_STRIDE, 1),
    (width, height),
)
if not writer.isOpened():
    cap.release()
    raise RuntimeError('Could not open VideoWriter for rigged character output.')

rig_base_height = height * CHARACTER_HEIGHT_FRACTION
_preview_target_x, _preview_target_bottom_y = _fixed_character_anchor(width, height)
print(
    f'colored character placement mode={CHARACTER_PLACEMENT_MODE} '
    f'fixed_position={CHARACTER_POSITION} fixed_x_fraction={CHARACTER_X_FRACTION} '
    f'fixed_target_x={_preview_target_x} fixed_bottom_y={_preview_target_bottom_y} '
    f'height_px={rig_base_height:.1f}'
)
print(
    f'foot_lock={CHARACTER_FOOT_LOCK_ENABLED} '
    f'anchor_x_smoothing={CHARACTER_ANCHOR_X_SMOOTHING} '
    f'anchor_y_smoothing={CHARACTER_ANCHOR_Y_SMOOTHING}'
)
print(
    f'partial_offscreen={CHARACTER_ALLOW_PARTIAL_OFFSCREEN_FRACTION} '
    f'floor_motion={CHARACTER_FLOOR_MOTION_ENABLED} floor_roi_bottom={CHARACTER_FLOOR_ROI_BOTTOM_FRACTION} '
    f'floor_every_n={CHARACTER_FLOOR_MOTION_EVERY_N_FRAMES} analysis_width={CHARACTER_FLOOR_MOTION_ANALYSIS_WIDTH}'
)
if CHARACTER_PLACEMENT_MODE == 'world_locked':
    print(
        '配置: world_locked。初期位置は人物相対(CHARACTER_RELATIVE_X/Y_OFFSET)で決め、'
        '以後は床・壁特徴点のカメラ運動推定(ホモグラフィ)で現実空間の床に固定します。'
        f' reattach_gain={CHARACTER_WORLD_LOCK_REATTACH_GAIN}'
        f' scale_with_camera={CHARACTER_WORLD_SCALE_WITH_CAMERA}'
        f' analysis_width={CHARACTER_WORLD_MOTION_ANALYSIS_WIDTH}'
        f' max_corners={CHARACTER_WORLD_MOTION_MAX_CORNERS}'
    )

elif CHARACTER_PLACEMENT_MODE == 'relative_to_person':
    print(
        '配置: 選択されたtrack人物のbboxに相対配置します（負のx_offsetは左側、負のy_offsetは奥/上側）。'
        f' x_offset={CHARACTER_RELATIVE_X_OFFSET} y_offset={CHARACTER_RELATIVE_Y_OFFSET}'
    )
elif CHARACTER_X_FRACTION is not None:
    print('注意: CHARACTER_X_FRACTION が指定されているため、CHARACTER_POSITION より手動横位置が優先されています。')
_preview_frame = next((frame for frame in character_motion_by_frame.values() if frame.get('people')), None)
_preview_person = _select_character_motion_person(_preview_frame, CHARACTER_SOURCE_PERSON, track_id=CHARACTER_SOURCE_TRACK_ID) if _preview_frame else None
if _preview_person is not None:
    print(
        'character source person: '
        f"selector={CHARACTER_SOURCE_PERSON} track_id={CHARACTER_SOURCE_TRACK_ID} frame={_preview_frame.get('frame_index')} "
        f"person_index={_preview_person.get('person_index')} track_id={_preview_person.get('track_id')} "
        f"conf={_preview_person.get('detection_confidence')} bbox={_preview_person.get('bbox_xyxy')}"
    )
else:
    print(f'character source person: selector={CHARACTER_SOURCE_PERSON} track_id={CHARACTER_SOURCE_TRACK_ID} no person found in first populated frame')

frame_idx = -1
rigged_written = 0
rigged_overlay_drawn = 0
rigged_no_person_frames = 0
rigged_no_keypoint_frames = 0
rigged_track_missing_frames = 0
rigged_hold_frames = 0
last_pose = None
last_overlay_bgra = None
last_mapper_state = None
last_anchor_state = None
last_floor_gray = None
last_floor_scale = 1.0
last_selected_person = None
dancer_seg_device_arg = 0 if torch.cuda.is_available() else 'cpu'
last_dancer_mask_alpha = None
dancer_occlusion_frames = 0
last_world_gray = None
world_anchor_state = None
world_h_missing_frames = 0
mp3d_used_frames = 0
last_z_map = None

progress = tqdm(
    total=len(character_motion_by_frame) if character_motion_by_frame else None,
    desc='colored articulated character render',
)

while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    frame_idx += 1
    if frame_idx not in character_motion_by_frame:
        continue

    frame_motion = character_motion_by_frame[frame_idx]
    person = _select_tracked_character_person(frame_motion, CHARACTER_SOURCE_PERSON, last_selected_person, track_id=CHARACTER_SOURCE_TRACK_ID)
    floor_gray = None
    floor_scale = last_floor_scale
    floor_dx, floor_dy = 0.0, 0.0
    run_floor_motion = (
        bool(CHARACTER_FLOOR_MOTION_ENABLED)
        and CHARACTER_PLACEMENT_MODE != 'world_locked'  # world_lockedはホモグラフィ追従が代替
        and (frame_idx % max(1, int(CHARACTER_FLOOR_MOTION_EVERY_N_FRAMES)) == 0)
    )
    if run_floor_motion:
        floor_gray, floor_scale = _downsample_gray_for_character_floor(frame_bgr, width)
        floor_dx, floor_dy = _estimate_floor_motion(last_floor_gray, floor_gray, person, width, height, motion_scale=floor_scale)
    world_H = None
    if CHARACTER_PLACEMENT_MODE == 'world_locked':
        world_gray, world_gray_scale = _downsample_gray_for_world(frame_bgr, width)
        world_H = _estimate_world_homography(last_world_gray, world_gray, person, motion_scale=world_gray_scale)
        if last_world_gray is not None and world_H is None:
            world_h_missing_frames += 1
        last_world_gray = world_gray

    if person is None:
        rigged_no_person_frames += 1
        if CHARACTER_SOURCE_TRACK_ID is not None:
            rigged_track_missing_frames += 1
        if last_overlay_bgra is not None and rigged_hold_frames < int(CHARACTER_MAX_HOLD_FRAMES):
            rendered = _alpha_blend_bgra(frame_bgr, last_overlay_bgra, CHARACTER_OPACITY)
            rigged_overlay_drawn += 1
            rigged_hold_frames += 1
        else:
            rendered = frame_bgr
            last_overlay_bgra = None
            last_pose = None
            last_mapper_state = None
            last_anchor_state = None
            last_selected_person = None
            rigged_hold_frames = 0
        if floor_gray is not None:
            last_floor_gray = floor_gray
            last_floor_scale = floor_scale
        writer.write(rendered)
        rigged_written += 1
        progress.update(1)
        continue

    last_selected_person = person
    _update_floor_level(person)
    kp_dict = _keypoints_dict_from_person(person)
    z_map = None
    if CHARACTER_USE_MP3D and MP3D_BY_FRAME:
        _mp_rec = MP3D_BY_FRAME.get(frame_idx)
        if _mp_rec:
            _kp_mp, _z_mp = _kp_dict_from_mp3d(_mp_rec)
            if len(_kp_mp) >= 3:
                kp_dict = _kp_mp
                z_map = _z_mp
                mp3d_used_frames += 1
    if z_map is not None:
        z_map = _smooth_z_map(z_map, last_z_map)
        last_z_map = z_map
    else:
        last_z_map = None

    if len(kp_dict) < 3:
        rigged_no_keypoint_frames += 1
        if last_overlay_bgra is not None and rigged_hold_frames < int(CHARACTER_MAX_HOLD_FRAMES):
            rendered = _alpha_blend_bgra(frame_bgr, last_overlay_bgra, CHARACTER_OPACITY)
            rigged_overlay_drawn += 1
            rigged_hold_frames += 1
        else:
            rendered = frame_bgr
            last_overlay_bgra = None
            last_pose = None
            last_mapper_state = None
            last_anchor_state = None
            last_selected_person = None
            rigged_hold_frames = 0
        if floor_gray is not None:
            last_floor_gray = floor_gray
            last_floor_scale = floor_scale
        writer.write(rendered)
        rigged_written += 1
        progress.update(1)
        continue

    if CHARACTER_PLACEMENT_MODE == 'world_locked':
        # 床・壁特徴点によるカメラ運動追従。人物bboxには追従しないため足滑りが起きません。
        rig_target_x, rig_target_bottom_y, rig_height_current, world_anchor_state = _update_world_locked_anchor(
            world_anchor_state, world_H, person, width, height, rig_base_height
        )
    else:
        rig_height_current = rig_base_height
        raw_rig_target_x, raw_rig_target_bottom_y = _character_anchor_for_person(width, height, person, rig_base_height)
        anchor_reference_state = _apply_floor_motion_to_anchor(last_anchor_state, floor_dx, floor_dy)
        rig_target_x, rig_target_bottom_y, last_anchor_state = _stabilize_character_anchor(
            raw_rig_target_x,
            raw_rig_target_bottom_y,
            anchor_reference_state,
            rig_base_height,
        )
    mapped_pose, mapper_state = _map_pose_for_character(
        kp_dict,
        person,
        rig_target_x,
        rig_target_bottom_y,
        rig_height_current,
        previous_pose=last_pose,
        previous_mapper_state=last_mapper_state,
    )

    if mapped_pose is None:
        rendered = frame_bgr
    else:
        last_pose = mapped_pose
        last_mapper_state = mapper_state
        overlay_bgra = _draw_colored_character(frame_bgr.shape, mapped_pose, rig_height_current, z_map=z_map)
        if overlay_bgra[:, :, 3].max() > 0:
            last_overlay_bgra = overlay_bgra
            rendered = _alpha_blend_bgra(frame_bgr, overlay_bgra, CHARACTER_OPACITY)
            rigged_overlay_drawn += 1
            rigged_hold_frames = 0
        else:
            rendered = frame_bgr

    # --- Dancer occlusion: 人形を描いた後、切り抜いたダンサーを最前面へ合成し直す ---
    if DANCER_OCCLUSION_ENABLED and dancer_seg_model is not None:
        if frame_idx % max(1, int(DANCER_SEG_EVERY_N_FRAMES)) == 0:
            seg_result = dancer_seg_model.predict(
                frame_bgr,
                conf=DANCER_SEG_CONF_THRESHOLD,
                imgsz=DANCER_SEG_IMGSZ,
                retina_masks=DANCER_SEG_RETINA_MASKS,
                classes=[0],
                device=dancer_seg_device_arg,
                verbose=False,
            )[0]
            last_dancer_mask_alpha = _select_dancer_mask_for_person(
                seg_result, person.get('bbox_xyxy'), width, height, DANCER_SEG_MATCH_IOU_MIN
            )
        if last_dancer_mask_alpha is not None:
            rendered = (
                frame_bgr.astype(np.float32) * last_dancer_mask_alpha
                + rendered.astype(np.float32) * (1.0 - last_dancer_mask_alpha)
            ).astype(np.uint8)
            dancer_occlusion_frames += 1

    if CHARACTER_DRAW_SOURCE_DEBUG:
        bbox = person.get('bbox_xyxy') if person else None
        if bbox and len(bbox) >= 4:
            x1, y1, x2, y2 = [int(v) for v in bbox[:4]]
            cv2.rectangle(rendered, (x1, y1), (x2, y2), (0, 255, 255), 2)

    if floor_gray is not None:
        last_floor_gray = floor_gray
        last_floor_scale = floor_scale
    writer.write(rendered)
    rigged_written += 1
    progress.update(1)

progress.close()
cap.release()
writer.release()

print(f'rigged_raw={rigged_raw_path}')
print(f'frames_written={rigged_written}')
print(f'frames_with_colored_character={rigged_overlay_drawn}')
print(f'frames_without_person={rigged_no_person_frames}')
print(f'frames_without_enough_keypoints={rigged_no_keypoint_frames}')
print(f'frames_with_requested_track_missing={rigged_track_missing_frames}')
print(f'held_previous_overlay_frames={rigged_hold_frames}')
if CHARACTER_FLOOR_LEVEL_ENABLED:
    _fy = _estimated_floor_y()
    print(f'estimated_floor_y={_fy if _fy is not None else "n/a"} (samples={len(FLOOR_LEVEL_STATE["samples"])})')
if DANCER_OCCLUSION_ENABLED:
    print(f'frames_with_dancer_drawn_above_puppet={dancer_occlusion_frames}')
if CHARACTER_PLACEMENT_MODE == 'world_locked':
    print(f'world_lock_frames_without_homography={world_h_missing_frames}')
    print('ヒント: 上の値が大きい場合、背景が単調(特徴点不足)です。CHARACTER_WORLD_MOTION_ANALYSIS_WIDTH を 960 へ上げるか、reattach_gain を 0.03 へ上げてください。')
if MP3D_BY_FRAME:
    print(f'frames_rendered_with_mediapipe_3d={mp3d_used_frames} (深度ソート・腰のくねり有効)')

if rigged_overlay_drawn == 0:
    print(
        'No colored character was drawn. Check that character_motion_by_frame contains '
        'people[*].keypoints with joint_name/x/y/keypoint_confidence.'
    )
else:
    print('Run the next transcode/display cell to create the audio-attached MP4 preview.')


In [ ]:
from IPython.display import HTML, display, Video
from base64 import b64encode
max_inline_mb = globals().get('max_inline_mb', 80)
if 'transcode_with_progress' not in globals():
    import subprocess
    from tqdm.auto import tqdm
    def transcode_with_progress(input_path, output_path, expected_duration_s=None):
        input_path = Path(input_path)
        output_path = Path(output_path)
        if expected_duration_s is None:
            cap_probe = cv2.VideoCapture(str(input_path))
            frame_count = cap_probe.get(cv2.CAP_PROP_FRAME_COUNT) or 0
            fps_probe = cap_probe.get(cv2.CAP_PROP_FPS) or 0
            cap_probe.release()
            expected_duration_s = float(frame_count / fps_probe) if fps_probe else None
        source_audio_path = Path(globals().get('video_path', input_path))
        command = ['ffmpeg', '-y', '-hide_banner', '-i', str(input_path), '-i', str(source_audio_path), '-map', '0:v:0', '-map', '1:a:0?', '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-shortest', '-progress', 'pipe:1', '-nostats', str(output_path)]
        progress = tqdm(total=expected_duration_s, unit='s', desc='ffmpeg transcode')
        last_time = 0.0
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            assert process.stdout is not None
            for line in process.stdout:
                line = line.strip()
                if line.startswith('out_time_ms='):
                    out_time_s = int(line.split('=', 1)[1]) / 1_000_000.0
                    if expected_duration_s:
                        progress.update(max(0.0, min(out_time_s, expected_duration_s) - last_time))
                    last_time = out_time_s
                elif line.startswith('progress=end') and expected_duration_s:
                    progress.update(max(0.0, expected_duration_s - progress.n))
            return_code = process.wait()
        finally:
            progress.close()
        if return_code != 0:
            raise RuntimeError(f'ffmpeg failed with exit code {return_code}')
        print(f'transcoded={output_path} size_mb={output_path.stat().st_size / 1024 / 1024:.1f}')

if 'rigged_raw_path' not in globals() or not Path(rigged_raw_path).exists():
    raise RuntimeError('色付き関節キャラクター動画がまだありません。先に色付き関節キャラクター合成セルを実行してください。')

rigged_raw_path = Path(rigged_raw_path)
rigged_final_path = OUTPUT_DIR / 'in_frame_character_rigged.mp4'
print(f'色付き関節キャラクター合成結果を音声付きMP4へ変換します: {rigged_raw_path}')
print('音声は元動画から保持します: ffmpeg -map 1:a:0? -c:a aac')
transcode_with_progress(rigged_raw_path, rigged_final_path, expected_duration_s=rigged_written / (fps / max(FRAME_STRIDE, 1)) if rigged_written else None)

rigged_size_mb = rigged_final_path.stat().st_size / 1024 / 1024
if rigged_size_mb <= max_inline_mb:
    print('embedding colored articulated character video preview...')
    rigged_video_b64 = b64encode(rigged_final_path.read_bytes()).decode('ascii')
    display(HTML(f'''<video width="960" controls><source src="data:video/mp4;base64,{rigged_video_b64}" type="video/mp4"></video>'''))
else:
    print(f'video is {rigged_size_mb:.1f} MB, skipping inline Base64 preview. Use the file path or final zip download.')
    display(Video(str(rigged_final_path), embed=False, width=960))
print(rigged_final_path)

## 14. 外部リグ/3D向けモーションを書き出す

にけえもんのような一体型イラストは、ノートブック内で無理に2D合成するより、Live2D・Spine・Blender・VRMなどでリグ化してから踊らせる方が自然です。次のセルは、選択したtrack IDのキーポイントをJSON/CSV/Blender補助スクリプトとして書き出します。

In [ ]:
# --- Export tracked motion for external rig / 3D / Live2D workflows
# にけえもんのような一体型イラストは、このノートブック内の2D合成より、Live2D/Spine/Blender/VRM側でリグ化して踊らせる方が自然です。
# このセルは、選択したtrack IDのキーポイントを外部ツールへ渡しやすいJSON/CSVに書き出します。

from pathlib import Path
import csv
import json

if 'motion_payload' not in globals():
    raise RuntimeError('motion_payload がありません。先にキーポイント解析・motion_payload生成セルを実行してください。')
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = Path('/content/rfdetr_keypoint_outputs')
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Export settings / 書き出し設定
# ------------------------------------------------------------------
# 書き出す人物のtrack ID。
# Noneの場合は CHARACTER_SOURCE_TRACK_ID を引き継ぎます。それもNoneなら各フレームの最大bbox人物を使います。
# 例: annotated_keypoints_yolo.mp4 で track 4 が対象なら MOTION_EXPORT_TRACK_ID = 4。
MOTION_EXPORT_TRACK_ID = globals().get('MOTION_EXPORT_TRACK_ID', globals().get('CHARACTER_SOURCE_TRACK_ID', None))
# track ID未指定時の人物選択。'largest' / 'highest_confidence' / 'rightmost' / 'leftmost' / 0,1,2...
MOTION_EXPORT_PERSON_SELECTOR = globals().get('MOTION_EXPORT_PERSON_SELECTOR', globals().get('CHARACTER_SOURCE_PERSON', 'largest'))
# 座標の原点変換。外部ツールでは正規化座標も一緒に使うのが安全です。
# pixel: 元動画ピクセル座標、normalized: 0-1座標、centered: 画面中央原点・高さ1.0基準。
MOTION_EXPORT_INCLUDE_CENTERED = globals().get('MOTION_EXPORT_INCLUDE_CENTERED', True)
# ファイル名の接頭辞。
MOTION_EXPORT_PREFIX = globals().get('MOTION_EXPORT_PREFIX', 'tracked_character_motion')

COCO_JOINTS = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle',
]

def _bbox_area(person):
    bbox = person.get('bbox_xyxy') if person else None
    if not bbox or len(bbox) < 4:
        return 0.0
    x1, y1, x2, y2 = [float(v) for v in bbox[:4]]
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def _select_export_person(frame_motion, selector='largest', track_id=None):
    people = list(frame_motion.get('people', [])) if frame_motion else []
    if not people:
        return None
    if track_id is not None:
        for person in people:
            if person.get('track_id') is not None and int(person.get('track_id')) == int(track_id):
                return person
        return None
    if selector == 'largest':
        return max(people, key=_bbox_area)
    if selector == 'highest_confidence':
        return max(people, key=lambda p: float(p.get('detection_confidence') or 0.0))
    if selector == 'rightmost':
        return max(people, key=lambda p: float((p.get('bbox_xyxy') or [0, 0, 0, 0])[2]))
    if selector == 'leftmost':
        return min(people, key=lambda p: float((p.get('bbox_xyxy') or [0, 0, 0, 0])[0]))
    if isinstance(selector, int) and 0 <= selector < len(people):
        return people[selector]
    return max(people, key=_bbox_area)

def _keypoints_by_name(person):
    result = {}
    for kp in person.get('keypoints', []) if person else []:
        name = kp.get('joint_name') or kp.get('name')
        if not name:
            continue
        result[name] = {
            'x': float(kp.get('x', 0.0)),
            'y': float(kp.get('y', 0.0)),
            'confidence': float(kp.get('keypoint_confidence', kp.get('confidence', 0.0)) or 0.0),
            'visible': bool(kp.get('visible', True)),
            'joint_index': kp.get('joint_index'),
        }
    return result

frame_width = int(motion_payload.get('frame_width', globals().get('width', 1)) or 1)
frame_height = int(motion_payload.get('frame_height', globals().get('height', 1)) or 1)
fps_export = float(motion_payload.get('fps', globals().get('fps', 30.0)) or 30.0)

selected_frames = []
missing_frames = 0
track_ids_seen = set()
for frame in motion_payload.get('frames', []):
    for person in frame.get('people', []):
        if person.get('track_id') is not None:
            track_ids_seen.add(int(person.get('track_id')))
    person = _select_export_person(frame, MOTION_EXPORT_PERSON_SELECTOR, track_id=MOTION_EXPORT_TRACK_ID)
    if person is None:
        missing_frames += 1
        selected_frames.append({
            'frame_index': int(frame.get('frame_index', -1)),
            'time_s': float(frame.get('time_s', 0.0)),
            'missing': True,
            'track_id': MOTION_EXPORT_TRACK_ID,
            'bbox_xyxy': None,
            'keypoints': {},
        })
        continue
    kps = _keypoints_by_name(person)
    normalized = {}
    centered = {}
    for name, kp in kps.items():
        x = float(kp['x'])
        y = float(kp['y'])
        normalized[name] = {
            'x': x / max(1.0, frame_width),
            'y': y / max(1.0, frame_height),
            'confidence': kp['confidence'],
            'visible': kp['visible'],
        }
        if MOTION_EXPORT_INCLUDE_CENTERED:
            centered[name] = {
                'x': (x - frame_width * 0.5) / max(1.0, frame_height),
                'y': -(y - frame_height * 0.5) / max(1.0, frame_height),
                'z': 0.0,
                'confidence': kp['confidence'],
                'visible': kp['visible'],
            }
    selected_frames.append({
        'frame_index': int(frame.get('frame_index', -1)),
        'time_s': float(frame.get('time_s', 0.0)),
        'missing': False,
        'person_index': person.get('person_index'),
        'track_id': person.get('track_id'),
        'detection_confidence': person.get('detection_confidence'),
        'bbox_xyxy': person.get('bbox_xyxy'),
        'keypoints': kps,
        'keypoints_normalized': normalized,
        'keypoints_centered': centered if MOTION_EXPORT_INCLUDE_CENTERED else {},
        'humanoid_landmarks': person.get('humanoid_landmarks', {}),
    })

export_payload = {
    'schema': 'er_flowscan.external_motion_export.v1',
    'description': '2D tracked pose export for Blender/Live2D/Spine/VRM retargeting.',
    'source_video': motion_payload.get('source_video'),
    'source_video_id': motion_payload.get('source_video_id'),
    'pose_source': motion_payload.get('pose_source'),
    'fps': fps_export,
    'frame_width': frame_width,
    'frame_height': frame_height,
    'selected_track_id': MOTION_EXPORT_TRACK_ID,
    'selector': MOTION_EXPORT_PERSON_SELECTOR,
    'track_ids_seen': sorted(track_ids_seen),
    'joint_order': COCO_JOINTS,
    'coordinate_spaces': {
        'pixel': 'x right, y down, original video pixels',
        'normalized': 'x/y in 0..1 from top-left',
        'centered': 'x/y centered on frame, scaled by frame height, y up, z=0',
    },
    'frames': selected_frames,
}

json_path = OUTPUT_DIR / f'{MOTION_EXPORT_PREFIX}.json'
json_path.write_text(json.dumps(export_payload, ensure_ascii=False, indent=2), encoding='utf-8')

long_csv_path = OUTPUT_DIR / f'{MOTION_EXPORT_PREFIX}_long.csv'
with long_csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'frame_index', 'time_s', 'missing', 'track_id', 'joint_name',
        'x', 'y', 'confidence', 'visible',
        'x_norm', 'y_norm', 'x_centered', 'y_centered', 'z_centered',
    ])
    writer.writeheader()
    for frame in selected_frames:
        for joint in COCO_JOINTS:
            kp = frame.get('keypoints', {}).get(joint)
            kpn = frame.get('keypoints_normalized', {}).get(joint)
            kpc = frame.get('keypoints_centered', {}).get(joint)
            writer.writerow({
                'frame_index': frame['frame_index'],
                'time_s': frame['time_s'],
                'missing': frame.get('missing', False),
                'track_id': frame.get('track_id'),
                'joint_name': joint,
                'x': kp.get('x') if kp else '',
                'y': kp.get('y') if kp else '',
                'confidence': kp.get('confidence') if kp else '',
                'visible': kp.get('visible') if kp else '',
                'x_norm': kpn.get('x') if kpn else '',
                'y_norm': kpn.get('y') if kpn else '',
                'x_centered': kpc.get('x') if kpc else '',
                'y_centered': kpc.get('y') if kpc else '',
                'z_centered': kpc.get('z') if kpc else '',
            })

wide_fields = ['frame_index', 'time_s', 'missing', 'track_id', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2']
for joint in COCO_JOINTS:
    wide_fields += [f'{joint}_x', f'{joint}_y', f'{joint}_conf', f'{joint}_x_norm', f'{joint}_y_norm', f'{joint}_x_centered', f'{joint}_y_centered']
wide_csv_path = OUTPUT_DIR / f'{MOTION_EXPORT_PREFIX}_wide.csv'
with wide_csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=wide_fields)
    writer.writeheader()
    for frame in selected_frames:
        bbox = frame.get('bbox_xyxy') or ['', '', '', '']
        row = {
            'frame_index': frame['frame_index'],
            'time_s': frame['time_s'],
            'missing': frame.get('missing', False),
            'track_id': frame.get('track_id'),
            'bbox_x1': bbox[0], 'bbox_y1': bbox[1], 'bbox_x2': bbox[2], 'bbox_y2': bbox[3],
        }
        for joint in COCO_JOINTS:
            kp = frame.get('keypoints', {}).get(joint, {})
            kpn = frame.get('keypoints_normalized', {}).get(joint, {})
            kpc = frame.get('keypoints_centered', {}).get(joint, {})
            row.update({
                f'{joint}_x': kp.get('x', ''),
                f'{joint}_y': kp.get('y', ''),
                f'{joint}_conf': kp.get('confidence', ''),
                f'{joint}_x_norm': kpn.get('x', ''),
                f'{joint}_y_norm': kpn.get('y', ''),
                f'{joint}_x_centered': kpc.get('x', ''),
                f'{joint}_y_centered': kpc.get('y', ''),
            })
        writer.writerow(row)

blender_helper_path = OUTPUT_DIR / f'{MOTION_EXPORT_PREFIX}_blender_import_empties.py'
blender_helper_path.write_text(r'''
# Blender helper: create animated empties from tracked_character_motion.json
# Usage in Blender: Text Editor > Open this file > set JSON_PATH > Run Script.
import json
import bpy
from pathlib import Path

JSON_PATH = r'tracked_character_motion.json'  # Change to the exported JSON path.
SCALE = 3.0
EMPTY_SIZE = 0.035

path = Path(JSON_PATH)
if not path.exists():
    raise FileNotFoundError(f'JSON not found: {path}')
with path.open('r', encoding='utf-8') as f:
    payload = json.load(f)
fps = float(payload.get('fps', 30.0))
bpy.context.scene.render.fps = int(round(fps))

joint_order = payload.get('joint_order', [])
objects = {}
for joint in joint_order:
    obj = bpy.data.objects.get(f'pose_{joint}')
    if obj is None:
        obj = bpy.data.objects.new(f'pose_{joint}', None)
        bpy.context.collection.objects.link(obj)
        obj.empty_display_type = 'SPHERE'
        obj.empty_display_size = EMPTY_SIZE
    objects[joint] = obj

for frame in payload.get('frames', []):
    if frame.get('missing'):
        continue
    frame_no = int(frame.get('frame_index', 0)) + 1
    centered = frame.get('keypoints_centered', {})
    for joint, coords in centered.items():
        obj = objects.get(joint)
        if obj is None or not coords:
            continue
        conf = float(coords.get('confidence', 0.0) or 0.0)
        if conf <= 0.05:
            continue
        obj.location = (float(coords.get('x', 0.0)) * SCALE, 0.0, float(coords.get('y', 0.0)) * SCALE)
        obj.keyframe_insert(data_path='location', frame=frame_no)

print(f'Imported pose empties: {len(objects)} joints, frames={len(payload.get("frames", []))}')
'''.strip() + '\n', encoding='utf-8')

readme_path = OUTPUT_DIR / f'{MOTION_EXPORT_PREFIX}_README.txt'
readme_path.write_text(f'''
Tracked motion export for external rig workflows

Files:
- {json_path.name}: full selected track motion. Recommended for Blender/Live2D/Spine conversion scripts.
- {long_csv_path.name}: one row per frame and joint. Easy to inspect/filter.
- {wide_csv_path.name}: one row per frame. Useful for spreadsheets or custom scripts.
- {blender_helper_path.name}: Blender helper that creates animated empties from the JSON.

Selected track ID: {MOTION_EXPORT_TRACK_ID}
Selector: {MOTION_EXPORT_PERSON_SELECTOR}
Track IDs seen: {sorted(track_ids_seen)}
FPS: {fps_export}
Frame size: {frame_width}x{frame_height}
Missing selected-person frames: {missing_frames} / {len(selected_frames)}

Recommended workflow for Nikemon:
1. Create or commission a proper Live2D/Spine/VRM/Blender rig from the illustration.
2. Use the exported JSON/CSV as motion source.
3. Retarget shoulder/elbow/wrist and hip/knee/ankle joints onto the rig controls.
4. Render the rigged character separately, then composite with the original video/audio.
'''.strip() + '\n', encoding='utf-8')

print('external motion export complete')
print(f'json={json_path}')
print(f'long_csv={long_csv_path}')
print(f'wide_csv={wide_csv_path}')
print(f'blender_helper={blender_helper_path}')
print(f'readme={readme_path}')
print(f'track_ids_seen={sorted(track_ids_seen)}')
print(f'selected_track_id={MOTION_EXPORT_TRACK_ID} missing_frames={missing_frames}/{len(selected_frames)}')

## 15. 出力をダウンロードする

必要な成果物をzip化します。

In [ ]:
SAVE_ZIP_TO_DRIVE = False
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/rfdetr_keypoint_outputs')

import shutil
from pathlib import Path

OUTPUT_DIR = Path(globals().get('OUTPUT_DIR', Path('/content/rfdetr_keypoint_outputs')))
if not OUTPUT_DIR.exists():
    raise RuntimeError(f'OUTPUT_DIR does not exist: {OUTPUT_DIR}')

zip_base = Path('/content') / OUTPUT_DIR.name
zip_path = Path(str(zip_base) + '.zip')
if zip_path.exists():
    zip_path.unlink()
archive_path = shutil.make_archive(str(zip_base), 'zip', root_dir=str(OUTPUT_DIR.parent), base_dir=OUTPUT_DIR.name)
zip_path = Path(archive_path)
print(f'created zip={zip_path} size_mb={zip_path.stat().st_size / 1024 / 1024:.1f}')

if SAVE_ZIP_TO_DRIVE:
    from google.colab import drive
    drive_mount_path = Path('/content/drive')
    if not drive_mount_path.exists() or not any(drive_mount_path.iterdir()):
        drive.mount('/content/drive', force_remount=False)
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    drive_zip_path = DRIVE_OUTPUT_DIR / zip_path.name
    shutil.copy2(zip_path, drive_zip_path)
    print(f'Copied zip to Drive: {drive_zip_path}')

from google.colab import files
files.download(str(zip_path))